In [ ]:
# In[1]:

import pandas as pd
import numpy as np
import pytz

# Ensure timezone variable for UTC+8 as required (not altering integer timestamps)
tz = pytz.timezone('Asia/Shanghai')

# Load CSV files into DataFrames (keep timestamps as integers)
df_metric = pd.read_csv('metric.csv', dtype={'timestamp': int})
df_trace  = pd.read_csv('trace.csv',  dtype={'timestamp': int})
df_log    = pd.read_csv('log.csv',    dtype={'timestamp': int})
df_err    = pd.read_csv('error_logs.csv', dtype={'timestamp': int})

# 1) metric.csv aggregation
metric_grp = df_metric.groupby(['cmdb_id', 'kpi_name'])['value']
metric_summary = metric_grp.agg(
    count = 'count',
    min   = 'min',
    p50   = lambda x: float(np.percentile(x.values, 50)),
    p95   = lambda x: float(np.percentile(x.values, 95)),
    max   = 'max'
).reset_index()
metric_summary_sorted = metric_summary.sort_values('count', ascending=False)
metric_summary_head = metric_summary_sorted[['cmdb_id','kpi_name','count','min','p50','p95','max']].head(20)

# 2) trace.csv aggregation
trace_grp = df_trace.groupby(['cmdb_id', 'trace_name'])['value']
trace_summary = trace_grp.agg(
    count = 'count',
    min   = 'min',
    p50   = lambda x: float(np.percentile(x.values, 50)),
    p95   = lambda x: float(np.percentile(x.values, 95)),
    max   = 'max'
).reset_index()
trace_summary_sorted = trace_summary.sort_values('count', ascending=False)
trace_summary_head = trace_summary_sorted[['cmdb_id','trace_name','count','min','p50','p95','max']].head(20)

# 3) log.csv aggregation
log_grp = df_log.groupby(['cmdb_id', 'log_name'])['value']
log_summary = log_grp.agg(
    count = 'count',
    min   = 'min',
    p50   = lambda x: float(np.percentile(x.values, 50)),
    p95   = lambda x: float(np.percentile(x.values, 95)),
    max   = 'max'
).reset_index()
log_summary_sorted = log_summary.sort_values('count', ascending=False)
log_summary_head = log_summary_sorted[['cmdb_id','log_name','count','min','p50','p95','max']].head(20)

# 4) error_logs.csv aggregation (timestamps remain integers)
err_grp = df_err.groupby('cmdb_id')['timestamp']
error_summary = err_grp.agg(
    count = 'count',
    earliest_ts = 'min',
    latest_ts = 'max'
).reset_index()
# Ensure integer dtype for earliest/latest
error_summary['earliest_ts'] = error_summary['earliest_ts'].astype(int)
error_summary['latest_ts'] = error_summary['latest_ts'].astype(int)
# Return all rows if small, otherwise top 20 by count
error_summary_sorted = error_summary.sort_values('count', ascending=False)
error_summary_head = error_summary_sorted[['cmdb_id','count','earliest_ts','latest_ts']].head(20)

# Display the compact summaries (up to 20 rows each)
metric_summary_head, trace_summary_head, log_summary_head, error_summary_head

```
Out[1]:
```
Summary of the aggregated telemetry (compact, top results):

Metric aggregates (top by sample count)
- Highest-count metric series have count = 61. Notable components/kpis:
  - shippingservice2-0: multiple container/network and mesh/source metrics (very large values for some kpis).
  - adservice: many app and runtime metrics (app.grpc.count, app.grpc.mrt, app.http.count, runtime.java_*), all with count = 61.
- For these series the reported p50/p95/min/max values are shown in the table (e.g., adservice app.grpc.count: min=247, p50=277, p95=299, max=313).

Trace aggregates (top by sample count)
- Top trace groups have count = 30. Notable components:
  - adservice-0 and adservice-1: multiple traces from frontend pods (duration_mean, duration_p95, error_rate, row_count).
  - shippingservice-1 and shippingservice2-0: traces from frontend as well.
- Example: adservice-0 trace.from_frontend-0.duration_mean (count=30, p50≈0.000018, p95≈0.000025).

Log aggregates (top by sample count)
- Top log groups have count = 31. Notable components:
  - checkoutservice-0: log.row_count and log.error_count (row_count p50=45, p95=65).
  - cartservice-0 and cartservice-2: log.row_count (very large row_count medians ≈590) and log.error_count (all zeros).
  - currencyservice-*, recommendationservice-*, emailservice-2, shippingservice-1 also appear with count=31.
- Many log.error_count series show min/p50/p95/max = 0 (no logged errors in those series).

Error logs
- Only two components have error log entries in the file (each with count=1):
  - checkoutservice-1: earliest/latest ts = 1647772098
  - frontend-2: earliest/latest ts = 1647772098

Overall takeaways
- adservice and shippingservice2-0 appear heavily represented in metric telemetry (most samples, many KPIs).
- adservice pods produce the most trace samples.
- Several service pods (checkoutservice, cartservice, currencyservice, recommendationservice) dominate log metrics (high row_count sampling); most log.error_count series are zero.
- Very few distinct components produced error_log entries (only checkoutservice-1 and frontend-2, one event each at timestamp 1647772098).

If you want, I can:
- Show the full top-20 tables again,
- Filter any of these summaries to specific time windows,
- Or drill into a specific component/KPI for more detail.

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id                                           kpi_name  count          min          p50          p95          max
5260  shippingservice2-0  mesh.source.shippingservice2.jaeger-collector....     61  2537.250000  3233.750000  3532.250000  3731.250000
0              adservice                                     app.grpc.count     61   247.000000   277.000000   299.000000   313.000000
1              adservice                                       app.grpc.mrt     61     0.002455     0.002550     0.002628     0.002677
2              adservice                                        app.grpc.rr     61   100.000000   100.000000   100.000000   100.000000
3              adservice                                        app.grpc.sr     61   100.000000   100.000000   100.000000   100.000000
4              adservice                                     app.http.count     61    24.000000    24.000000    24.000000    24.000000
5              adservice                                       app.http.mrt     61     0.008398     0.008769     0.008912     0.009079
6              adservice                                        app.http.rr     61   100.000000   100.000000   100.000000   100.000000
7              adservice                                        app.http.sr     61   100.000000   100.000000   100.000000   100.000000
8              adservice    runtime.java_lang_ClassLoading_LoadedClassCount     61  5162.000000  5162.000000  5162.000000  5162.000000
9              adservice  runtime.java_lang_ClassLoading_TotalLoadedClas...     61  5185.000000  5185.000000  5185.000000  5185.000000
10             adservice  runtime.java_lang_ClassLoading_UnloadedClassCount     61    23.000000    23.000000    23.000000    23.000000
11             adservice             runtime.java_lang_ClassLoading_Verbose     61     0.000000     0.000000     0.000000     0.000000
5221  shippingservice2-0  container.node-6.container_network_receive_MB....     61     0.048193     0.069576     0.358241     0.398905
5222  shippingservice2-0  container.node-6.container_network_receive_err...     61     0.000000     0.000000     0.000000     0.000000
5223  shippingservice2-0  container.node-6.container_network_receive_pac...     61   380.000000   523.000000   608.500000   663.000000
5224  shippingservice2-0  container.node-6.container_network_receive_pac...     61     0.000000     0.000000     0.000000     0.000000
5225  shippingservice2-0  container.node-6.container_network_transmit_MB...     61     0.514330     0.567779     0.589571     0.600361
5226  shippingservice2-0  container.node-6.container_network_transmit_er...     61     0.000000     0.000000     0.000000     0.000000
5227  shippingservice2-0  container.node-6.container_network_transmit_pa...     61   236.500000   326.000000   379.000000   422.500000,                  cmdb_id                           trace_name  count        min        p50        p95        max
1231  shippingservice2-0     trace.from_frontend2-0.row_count     30  31.000000  56.500000  76.400000  84.000000
0            adservice-0  trace.from_frontend-0.duration_mean     30   0.000015   0.000018   0.000025   0.000046
1            adservice-0   trace.from_frontend-0.duration_p95     30   0.000016   0.000024   0.000046   0.000114
2            adservice-0     trace.from_frontend-0.error_rate     30   0.000000   0.000000   0.000000   0.000000
3            adservice-0      trace.from_frontend-0.row_count     30   3.000000   7.000000  10.000000  10.000000
4            adservice-0  trace.from_frontend-1.duration_mean     30   0.000017   0.000019   0.000021   0.000024
5            adservice-0   trace.from_frontend-1.duration_p95     30   0.000020   0.000027   0.000036   0.000042
6            adservice-0     trace.from_frontend-1.error_rate     30   0.000000   0.000000   0.000000   0.000000
7            adservice-0      trace.from_frontend-1.row_count     30  17.000000  20.500000  24.000000  25.000000
8            adservice-0  trace.from_frontend-2.duration_mean     30   0.000016   0.000019   0.000024   0.000030
9            adservice-0   trace.from_frontend-2.duration_p95     30   0.000020   0.000025   0.000053   0.000077
10           adservice-0     trace.from_frontend-2.error_rate     30   0.000000   0.000000   0.000000   0.000000
11           adservice-0      trace.from_frontend-2.row_count     30   7.000000  10.000000  12.550000  13.000000
12           adservice-1  trace.from_frontend-0.duration_mean     30   0.000015   0.000017   0.000019   0.000021
13           adservice-1   trace.from_frontend-0.duration_p95     30   0.000016   0.000019   0.000027   0.000029
14           adservice-1     trace.from_frontend-0.error_rate     30   0.000000   0.000000   0.000000   0.000000
1192   shippingservice-1  trace.from_frontend-1.duration_mean     30   0.000068   0.000072   0.000118   0.000125
1193   shippingservice-1   trace.from_frontend-1.duration_p95     30   0.000072   0.000080   0.000280   0.000319
1194   shippingservice-1     trace.from_frontend-1.error_rate     30   0.000000   0.000000   0.000000   0.000000
1195   shippingservice-1      trace.from_frontend-1.row_count     30   3.000000   7.000000  10.100000  11.000000,                     cmdb_id         log_name  count   min    p50    p95    max
13        checkoutservice-0    log.row_count     31   3.0   45.0   65.0   68.0
12        checkoutservice-0  log.error_count     31   0.0    0.0    0.0    0.0
10            cartservice-2  log.error_count     31   0.0    0.0    0.0    0.0
11            cartservice-2    log.row_count     31  19.0  589.0  658.0  678.0
7             cartservice-0    log.row_count     31  18.0  590.0  658.5  669.0
6             cartservice-0  log.error_count     31   0.0    0.0    0.0    0.0
22        currencyservice-2  log.error_count     31   0.0    0.0    0.0    0.0
23        currencyservice-2    log.row_count     31   4.0  510.0  629.5  637.0
28           emailservice-2  log.error_count     31   0.0    0.0    0.0    0.0
29           emailservice-2    log.row_count     31   1.0    8.0   30.0   55.0
18        currencyservice-0  log.error_count     31   0.0    0.0    0.0    0.0
19        currencyservice-0    log.row_count     31   3.0  516.0  626.5  633.0
20        currencyservice-1  log.error_count     31   0.0    0.0    0.0    0.0
21        currencyservice-1    log.row_count     31   3.0  513.0  631.5  642.0
53  recommendationservice-2    log.row_count     31  21.0  146.0  161.5  165.0
52  recommendationservice-2  log.error_count     31   0.0    0.0    0.0    0.0
51  recommendationservice-1    log.row_count     31   1.0  146.0  164.0  170.0
50  recommendationservice-1  log.error_count     31   0.0    0.0    0.0    0.0
58        shippingservice-1  log.error_count     31   0.0    0.0    0.0    0.0
59        shippingservice-1    log.row_count     31   2.0   63.0   81.0   89.0,              cmdb_id  count  earliest_ts   latest_ts
0  checkoutservice-1      1   1647772098  1647772098
1         frontend-2      1   1647772098  1647772098)```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# reuse df_metric loaded previously in the kernel
# ensure tz for UTC+8
tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 and P5 for each (cmdb_id, kpi_name) using the full series
thresholds = df_metric.groupby(['cmdb_id', 'kpi_name'])['value'].agg(
    global_p95 = lambda x: float(np.percentile(x.values, 95)),
    global_p5  = lambda x: float(np.percentile(x.values, 5))
).reset_index()

# 2) Define incident window March 20, 2022 18:00 to 18:30 (UTC+8) and filter metric points into that window
start_local = tz.localize(datetime(2022, 3, 20, 18, 0, 0))
end_local   = tz.localize(datetime(2022, 3, 20, 18, 30, 0))
# convert to UTC epoch seconds for comparison with metric timestamps
start_ts = int(start_local.astimezone(pytz.utc).timestamp())
end_ts   = int(end_local.astimezone(pytz.utc).timestamp())

df_window = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()

# Merge thresholds into windowed points so we can detect anomalies
df_win_merged = df_window.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# Mark anomaly points: value >= global_p95 OR value <= global_p5
df_win_merged['is_anom'] = (df_win_merged['value'] >= df_win_merged['global_p95']) | (df_win_merged['value'] <= df_win_merged['global_p5'])

# 3) Aggregate anomaly points per series (only series with at least one anomaly point)
df_anom_points = df_win_merged[df_win_merged['is_anom']].copy()

anom_summary = df_anom_points.groupby(['cmdb_id', 'kpi_name']).agg(
    anomaly_count = ('value', 'count'),
    earliest_anomaly_ts = ('timestamp', 'min'),
    latest_anomaly_ts   = ('timestamp', 'max')
).reset_index()

# Also compute window-level min/max values per series (all points inside the window)
window_stats = df_window.groupby(['cmdb_id', 'kpi_name'])['value'].agg(
    window_min_value = 'min',
    window_max_value = 'max'
).reset_index()

# Merge summaries with thresholds and window stats
anom_agg = anom_summary.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left') \
                       .merge(window_stats, on=['cmdb_id', 'kpi_name'], how='left')

# Ensure integer timestamps
if not anom_agg.empty:
    anom_agg['earliest_anomaly_ts'] = anom_agg['earliest_anomaly_ts'].astype(int)
    anom_agg['latest_anomaly_ts']   = anom_agg['latest_anomaly_ts'].astype(int)

# Compact result: top 20 rows ordered by anomaly_count desc
anom_agg_sorted = anom_agg.sort_values('anomaly_count', ascending=False)
anom_agg_head = anom_agg_sorted[['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_ts','latest_anomaly_ts','window_min_value','window_max_value','global_p95','global_p5']].head(20)

# Display the compact aggregate table
anom_agg_head

```
Out[2]:
```
Summary of findings (metric anomalies in March 20, 2022 18:00–18:30 UTC+8):

- Top results (limited to 20) show multiple series with anomaly_count = 31 (i.e., anomaly at every sampled minute in the window). The earliest and latest anomaly timestamps span the full inspected window (1647770400 … 1647772200), which corresponds to the requested incident window.
- Dominant components:
  - shippingservice2-0: many container/node metrics flagged (network, fs, memory_cache, threads, ulimits, etc.).
  - adservice: several app and runtime metrics flagged (e.g., app.grpc.rr, app.grpc.sr, app.http.count, java class-loading metrics).
- A common pattern: global_p95 and global_p5 are equal to the series values (the series are effectively constant across the full history). Because the anomaly rule was value >= global_p95 OR value <= global_p5, constant series meet that condition for every point — hence the anomaly_count = full-window-length.
- Examples from the top table:
  - shippingservice2-0.container_ulimits_soft.max_open_files — window_min/max = 1.048576e+06, global_p95 = global_p5 = 1.048576e+06, anomaly_count = 31.
  - adservice.app.grpc.rr and app.grpc.sr — window value = 100, global_p95 = global_p5 = 100, anomaly_count = 31.
  - many shippingservice2-0 fs and write/read metrics are constant zeros and thus flagged (global_p95 = global_p5 = 0).
- Interpretation: These are not transient spikes — they are series with constant values that match their global percentiles. This is likely an artifact of the percentile-based rule on constant or configuration-like metrics rather than indicative of a sudden abnormal event.
- Suggested next steps: inspect the metric semantics (are these counters/config values?), exclude constant/config metrics from anomaly detection, or change anomaly criteria (e.g., require deviation from local baseline or require value strictly > p95 / < p5).

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  anomaly_count  earliest_anomaly_ts  latest_anomaly_ts  window_min_value  window_max_value    global_p95     global_p5
5197  shippingservice2-0  container.node-6.container_ulimits_soft.max_op...             31           1647770400         1647772200      1.048576e+06      1.048576e+06  1.048576e+06  1.048576e+06
5196  shippingservice2-0             container.node-6.container_threads_max             31           1647770400         1647772200      0.000000e+00      0.000000e+00  0.000000e+00  0.000000e+00
5195  shippingservice2-0                 container.node-6.container_threads             31           1647770400         1647772200      2.900000e+01      2.900000e+01  2.900000e+01  2.900000e+01
2              adservice                                        app.grpc.rr             31           1647770400         1647772200      1.000000e+02      1.000000e+02  1.000000e+02  1.000000e+02
3              adservice                                        app.grpc.sr             31           1647770400         1647772200      1.000000e+02      1.000000e+02  1.000000e+02  1.000000e+02
4              adservice                                     app.http.count             31           1647770400         1647772200      2.400000e+01      2.400000e+01  2.400000e+01  2.400000e+01
5194  shippingservice2-0  container.node-6.container_tasks_state.uninter...             31           1647770400         1647772200      0.000000e+00      0.000000e+00  0.000000e+00  0.000000e+00
6              adservice                                        app.http.rr             31           1647770400         1647772200      1.000000e+02      1.000000e+02  1.000000e+02  1.000000e+02
7              adservice                                        app.http.sr             31           1647770400         1647772200      1.000000e+02      1.000000e+02  1.000000e+02  1.000000e+02
8              adservice    runtime.java_lang_ClassLoading_LoadedClassCount             31           1647770400         1647772200      5.162000e+03      5.162000e+03  5.162000e+03  5.162000e+03
9              adservice  runtime.java_lang_ClassLoading_TotalLoadedClas...             31           1647770400         1647772200      5.185000e+03      5.185000e+03  5.185000e+03  5.185000e+03
5156  shippingservice2-0  container.node-6.container_fs_write_seconds./d...             31           1647770400         1647772200      0.000000e+00      0.000000e+00  0.000000e+00  0.000000e+00
5157  shippingservice2-0      container.node-6.container_fs_writes./dev/vda             31           1647770400         1647772200      0.000000e+00      0.000000e+00  0.000000e+00  0.000000e+00
5158  shippingservice2-0     container.node-6.container_fs_writes./dev/vda1             31           1647770400         1647772200      0.000000e+00      0.000000e+00  0.000000e+00  0.000000e+00
5159  shippingservice2-0   container.node-6.container_fs_writes_MB./dev/vda             31           1647770400         1647772200      0.000000e+00      0.000000e+00  0.000000e+00  0.000000e+00
5160  shippingservice2-0  container.node-6.container_fs_writes_merged./d...             31           1647770400         1647772200      0.000000e+00      0.000000e+00  0.000000e+00  0.000000e+00
5162  shippingservice2-0            container.node-6.container_memory_cache             31           1647770400         1647772200      2.349466e+07      2.349466e+07  2.349466e+07  2.349466e+07
5147  shippingservice2-0   container.node-6.container_fs_limit_MB./dev/vda1             31           1647770400         1647772200      6.046307e+05      6.046307e+05  6.046307e+05  6.046307e+05
5148  shippingservice2-0  container.node-6.container_fs_read_seconds./de...             31           1647770400         1647772200      0.000000e+00      0.000000e+00  0.000000e+00  0.000000e+00
5149  shippingservice2-0       container.node-6.container_fs_reads./dev/vda             31           1647770400         1647772200      0.000000e+00      1.000000e+00  0.000000e+00  0.000000e+00```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# reuse existing df_metric and thresholds if available; otherwise load/recompute
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv', dtype={'timestamp': int})

try:
    thresholds
except NameError:
    thresholds = df_metric.groupby(['cmdb_id', 'kpi_name'])['value'].agg(
        global_p95 = lambda x: float(np.percentile(x.values, 95)),
        global_p5  = lambda x: float(np.percentile(x.values, 5))
    ).reset_index()

# timezone and incident window (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
start_local = tz.localize(datetime(2022, 3, 20, 18, 0, 0))
end_local   = tz.localize(datetime(2022, 3, 20, 18, 30, 0))
start_ts = int(start_local.astimezone(pytz.utc).timestamp())
end_ts   = int(end_local.astimezone(pytz.utc).timestamp())

# 2) Exclude series where global_p95 == global_p5 (constant/config metrics)
thresholds_var = thresholds.copy()
variable_thresholds = thresholds_var[thresholds_var['global_p95'] != thresholds_var['global_p5']].copy()

# 3) Filter metric points to the incident window and keep only non-constant series
df_window = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()
df_win_merged = df_window.merge(variable_thresholds, on=['cmdb_id', 'kpi_name'], how='inner')

# Mark anomaly points: value >= global_p95 OR value <= global_p5
df_win_merged['is_anom'] = (df_win_merged['value'] >= df_win_merged['global_p95']) | (df_win_merged['value'] <= df_win_merged['global_p5'])

# 4) Aggregate anomalies per series
df_anom_points = df_win_merged[df_win_merged['is_anom']].copy()

if not df_anom_points.empty:
    anom_summary = df_anom_points.groupby(['cmdb_id', 'kpi_name']).agg(
        anomaly_count = ('value', 'count'),
        earliest_anomaly_ts = ('timestamp', 'min'),
        latest_anomaly_ts   = ('timestamp', 'max')
    ).reset_index()

    # window-level min/max for the same (cmdb_id, kpi_name) among all points in window (non-constant series)
    window_stats = df_win_merged.groupby(['cmdb_id', 'kpi_name'])['value'].agg(
        window_min_value = 'min',
        window_max_value = 'max'
    ).reset_index()

    # Merge summaries with thresholds and window stats
    anom_agg = anom_summary.merge(variable_thresholds, on=['cmdb_id', 'kpi_name'], how='left') \
                           .merge(window_stats, on=['cmdb_id', 'kpi_name'], how='left')

    # Ensure integer timestamps
    anom_agg['earliest_anomaly_ts'] = anom_agg['earliest_anomaly_ts'].astype(int)
    anom_agg['latest_anomaly_ts']   = anom_agg['latest_anomaly_ts'].astype(int)

    # Compact result: top 20 rows ordered by anomaly_count desc
    anom_agg_sorted = anom_agg.sort_values('anomaly_count', ascending=False)
    anom_agg_head = anom_agg_sorted[['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_ts','latest_anomaly_ts','window_min_value','window_max_value','global_p95','global_p5']].head(20)
else:
    # empty result frame with expected columns
    anom_agg_head = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_count','earliest_anomaly_ts','latest_anomaly_ts','window_min_value','window_max_value','global_p95','global_p5'])

# Display compact aggregate table (up to 20 rows)
anom_agg_head

```
Out[3]:
```
Summary of anomalies inside March 20, 2022 18:00–18:30 (UTC+8) after excluding constant/config series:

- Result overview:
  - Found multiple non-constant metric series with anomaly points in the window.
  - The compact top-20 table (shown earlier) contains series with anomaly_count = 31, meaning an anomaly at every sampled minute across the full 30-minute window (timestamps 1647770400 … 1647772200).

- Affected components and metric types (representative):
  - shippingservice-2 / frontend-2 / paymentservice-0 / currencyservice-* / recommendationservice-0 / cartservice2-0 / adservice-0 / node-6.
  - Many anomalies are container/node metrics: container_memory_mapped_file, container_memory_cache, container_fs_reads_MB, container_fs_writes, container_cpu_cfs_throttled_*, container_file_descriptors, container_threads, system.process.zombie.num, runtime.jvm_classes_loaded, etc.

- What the anomalies mean here:
  - These series were not constant (global_p95 != global_p5), and during the incident window individual per-minute values either exceeded the global P95 or fell below the global P5.
  - For the top entries the anomalies were persistent (present for every minute in the window), indicating a sustained deviation relative to the series' historical distribution.
  - Example specifics:
    - adservice-0.container_threads: window values ranged 29–30, global_p95 ≈ 29.5, global_p5 = 29.0 → values hit the P95 or fell to P5 threshold each minute.
    - paymentservice-0.container_file_descriptors: window 21–31, global_p95 = 31, global_p5 = 21 → values hit extremes through the window.
    - currencyservice-* and recommendationservice-* show sustained high memory_cache / memory_mapped_file values relative to their historical percentiles.
    - node-6.system.process.zombie.num: window 253–262, global_p95 = 262, global_p5 = 253 → sustained high zombie process counts near historic extremes.

- Practical interpretation and next steps:
  - These are sustained deviations (not isolated spikes). Investigate semantics per metric to decide severity (e.g., high zombie process count or file-descriptor saturation are actionable; small shifts in thread counts or mapped-file sizes may be benign).
  - For each high-priority series, inspect the original time series, check whether anomalies are above P95 or below P5, correlate with logs/traces, and verify whether system behavior (errors, latency) coincides.
  - If many series are at historical extremes by design (e.g., load test or deployment), consider using a rolling/local baseline or stricter anomaly rules to reduce persistent alerts.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  anomaly_count  earliest_anomaly_ts  latest_anomaly_ts  window_min_value  window_max_value    global_p95   global_p5
61                adservice-0                 container.node-6.container_threads             31           1647770400         1647772200              29.0      3.000000e+01  2.950000e+01        29.0
1778        shippingservice-2      container.node-5.container_memory_mapped_file             31           1647770400         1647772200         4190208.0      4.190208e+06  4.460544e+06   4190208.0
31                  adservice                         runtime.jvm_classes_loaded             31           1647770400         1647772200            5162.0      5.162000e+03  5.162000e+03         0.0
1304         paymentservice-0        container.node-6.container_file_descriptors             31           1647770400         1647772200              21.0      3.100000e+01  3.100000e+01        21.0
644         currencyservice-2  container.node-6.container_cpu_cfs_throttled_s...             31           1647770400         1647772200               0.0      2.281554e-02  1.840895e-02         0.0
955                frontend-2    container.node-5.container_fs_reads_MB./dev/vda             31           1647770400         1647772200               0.0      1.367188e-01  5.078125e-02         0.0
961                frontend-2      container.node-5.container_memory_mapped_file             31           1647770400         1647772200         2838528.0      2.838528e+06  2.973696e+06   2838528.0
622         currencyservice-1      container.node-6.container_memory_mapped_file             31           1647770400         1647772200         4866048.0      5.001216e+06  5.001216e+06   4866048.0
590         currencyservice-0            container.node-6.container_memory_cache             31           1647770400         1647772200        15495168.0      1.566720e+07  1.566720e+07  15495168.0
619         currencyservice-1            container.node-6.container_memory_cache             31           1647770400         1647772200        14962688.0      1.524531e+07  1.524531e+07  14962688.0
1597  recommendationservice-0            container.node-6.container_memory_cache             31           1647770400         1647772200        33492992.0      3.380429e+07  3.380429e+07  33492992.0
593         currencyservice-0      container.node-6.container_memory_mapped_file             31           1647770400         1647772200         8921088.0      9.056256e+06  9.056256e+06   8921088.0
650         currencyservice-2            container.node-6.container_memory_cache             31           1647770400         1647772200        13512704.0      1.364787e+07  1.364787e+07  13512704.0
1307         paymentservice-0   container.node-6.container_fs_writes_MB./dev/vda             31           1647770400         1647772200               0.0      8.593750e-02  1.367188e-02         0.0
1306         paymentservice-0      container.node-6.container_fs_writes./dev/vda             31           1647770400         1647772200               0.0      8.000000e+00  1.500000e+00         0.0
1312         paymentservice-0      container.node-6.container_memory_mapped_file             31           1647770400         1647772200         1622016.0      1.757184e+06  1.757184e+06   1622016.0
1294                   node-6                          system.process.zombie.num             31           1647770400         1647772200             253.0      2.620000e+02  2.620000e+02       253.0
303            cartservice2-0    container.node-5.container_fs_reads_MB./dev/vda             31           1647770400         1647772200               0.0      8.632812e-01  3.906250e-01         0.0
297            cartservice2-0  container.node-5.container_cpu_cfs_throttled_p...             31           1647770400         1647772200               0.0      3.000000e+00  1.000000e+00         0.0
643         currencyservice-2  container.node-6.container_cpu_cfs_throttled_p...             31           1647770400         1647772200               0.0      1.000000e+00  5.000000e-01         0.0```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# Reuse df_trace if present, else load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv', dtype={'timestamp': int})

# Ensure timestamp is integer
df_trace['timestamp'] = df_trace['timestamp'].astype(int)

# 1) Compute global P95 and P5 per (cmdb_id, trace_name) using full series
trace_thresholds = df_trace.groupby(['cmdb_id', 'trace_name'])['value'].agg(
    global_p95 = lambda x: float(np.percentile(x.values, 95)),
    global_p5  = lambda x: float(np.percentile(x.values, 5))
).reset_index()

# 2) Define incident window March 20, 2022 18:00 to 18:30 (UTC+8) and filter trace points into that window
tz = pytz.timezone('Asia/Shanghai')
start_local = tz.localize(datetime(2022, 3, 20, 18, 0, 0))
end_local   = tz.localize(datetime(2022, 3, 20, 18, 30, 0))
start_ts = int(start_local.astimezone(pytz.utc).timestamp())
end_ts   = int(end_local.astimezone(pytz.utc).timestamp())

df_trace_window = df_trace[(df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts)].copy()

# 3) Merge thresholds into windowed traces and mark anomalies: value >= global_p95 OR value <= global_p5
df_trace_win_merged = df_trace_window.merge(trace_thresholds, on=['cmdb_id', 'trace_name'], how='left')
df_trace_win_merged['is_anom'] = (df_trace_win_merged['value'] >= df_trace_win_merged['global_p95']) | \
                                 (df_trace_win_merged['value'] <= df_trace_win_merged['global_p5'])

# 4A) Aggregate anomalies per (cmdb_id, trace_name)
df_trace_anom_points = df_trace_win_merged[df_trace_win_merged['is_anom']].copy()

if not df_trace_anom_points.empty:
    trace_anom_summary = df_trace_anom_points.groupby(['cmdb_id', 'trace_name']).agg(
        anomaly_count = ('value', 'count'),
        earliest_anomaly_ts = ('timestamp', 'min'),
        latest_anomaly_ts   = ('timestamp', 'max')
    ).reset_index()

    # window-level min/max among all points in the window for those series
    window_stats = df_trace_window.groupby(['cmdb_id', 'trace_name'])['value'].agg(
        window_min_value = 'min',
        window_max_value = 'max'
    ).reset_index()

    # Merge summaries with thresholds and window stats
    trace_anom_agg = trace_anom_summary.merge(trace_thresholds, on=['cmdb_id', 'trace_name'], how='left') \
                                       .merge(window_stats, on=['cmdb_id', 'trace_name'], how='left')

    # Ensure integer timestamps
    trace_anom_agg['earliest_anomaly_ts'] = trace_anom_agg['earliest_anomaly_ts'].astype(int)
    trace_anom_agg['latest_anomaly_ts']   = trace_anom_agg['latest_anomaly_ts'].astype(int)

    # Compact A: top 20 by anomaly_count desc
    trace_anom_sorted = trace_anom_agg.sort_values('anomaly_count', ascending=False)
    trace_anom_head = trace_anom_sorted[['cmdb_id','trace_name','anomaly_count','earliest_anomaly_ts','latest_anomaly_ts','window_min_value','window_max_value','global_p95','global_p5']].head(20)
else:
    trace_anom_agg = pd.DataFrame(columns=['cmdb_id','trace_name','anomaly_count','earliest_anomaly_ts','latest_anomaly_ts','global_p95','global_p5','window_min_value','window_max_value'])
    trace_anom_head = trace_anom_agg.head(0)

# 4B) From A) prioritize trace_names containing 'error_rate' or 'duration' or 'row_count' (case-sensitive)
mask_callq = trace_anom_sorted['trace_name'].str.contains('error_rate|duration|row_count', regex=True)
trace_anom_callq = trace_anom_sorted[mask_callq][['cmdb_id','trace_name','anomaly_count','earliest_anomaly_ts','latest_anomaly_ts','window_min_value','window_max_value','global_p95','global_p5']].head(50)

# Final compact outputs (stored in kernel variables; displayed as the result)
trace_anom_head, trace_anom_callq

```
Out[4]:
```
Summary of trace anomalies inside March 20, 2022 18:00–18:30 (UTC+8):

- What was found (compact):
  - Many trace series show "anomalies" for the entire inspected window (anomaly_count = 30; earliest_ts = 1647770400, latest_ts = 1647772140).
  - The top-20 anomalous series (by anomaly_count) include many error_rate metrics from frontend→service calls and some self/duration/row_count traces across services (e.g., adservice-0/1, frontend-0/1, cartservice-1, shippingservice2-0).
  - Example rows: adservice-1.trace.from_frontend-0.error_rate, frontend-0.trace.to_cartservice-0.error_rate, shippingservice2-0.trace.from_frontend2-0.error_rate — each with window values = 0.0 and global_p95 = global_p5 = 0.0.

- Prioritized call-quality traces (filtered for 'error_rate' / 'duration' / 'row_count'):
  - The filtered list (up to 50 rows) is dominated by error_rate traces (many frontend→service error_rate series).
  - All these prioritized traces reported window_min = window_max = 0.0 and global_p95 = global_p5 = 0.0 (so every point equals the global thresholds).

- Interpretation:
  - The detected “anomalies” are mostly constant-zero series where per-minute values equal the global p5/p95 thresholds. Because the test used value >= p95 or <= p5, constant series match that rule and appear anomalous for every sample.
  - This pattern likely reflects benign constant telemetry (no errors observed) or metric semantics (sparse/zero error rates), not necessarily problematic behavior.

- Suggested next steps:
  - Exclude constant series (global_p95 == global_p5) before flagging, or require strict inequality (value > p95 or value < p5).
  - Focus investigation on traces with non-zero error_rate spikes or elevated duration_p95 values; correlate those with logs/metrics if present.
  - If you want, I can re-run the same trace anomaly query after excluding constant series, or list the few traces that actually deviate (non-constant) in the window.

The original code execution output of IPython Kernel is also provided below for reference:

(                 cmdb_id                                   trace_name  anomaly_count  earliest_anomaly_ts  latest_anomaly_ts  window_min_value  window_max_value  global_p95  global_p5
14           adservice-1             trace.from_frontend-0.error_rate             30           1647770400         1647772140               0.0               0.0         0.0        0.0
1230  shippingservice2-0            trace.from_frontend2-0.error_rate             30           1647770400         1647772140               0.0               0.0         0.0        0.0
10           adservice-0             trace.from_frontend-2.error_rate             30           1647770400         1647772140               0.0               0.0         0.0        0.0
22           adservice-1             trace.from_frontend-2.error_rate             30           1647770400         1647772140               0.0               0.0         0.0        0.0
2            adservice-0             trace.from_frontend-0.error_rate             30           1647770400         1647772140               0.0               0.0         0.0        0.0
18           adservice-1             trace.from_frontend-1.error_rate             30           1647770400         1647772140               0.0               0.0         0.0        0.0
590           frontend-0            trace.to_cartservice-0.error_rate             30           1647770400         1647772140               0.0               0.0         0.0        0.0
642           frontend-0  trace.to_recommendationservice-1.error_rate             30           1647770400         1647772140               0.0               0.0         0.0        0.0
646           frontend-0  trace.to_recommendationservice-2.error_rate             30           1647770400         1647772140               0.0               0.0         0.0        0.0
634           frontend-0  trace.to_productcatalogservice-2.error_rate             30           1647770400         1647772140               0.0               0.0         0.0        0.0
638           frontend-0  trace.to_recommendationservice-0.error_rate             30           1647770400         1647772140               0.0               0.0         0.0        0.0
594           frontend-0            trace.to_cartservice-1.error_rate             30           1647770400         1647772140               0.0               0.0         0.0        0.0
598           frontend-0            trace.to_cartservice-2.error_rate             30           1647770400         1647772140               0.0               0.0         0.0        0.0
586           frontend-0              trace.to_adservice-2.error_rate             30           1647770400         1647772140               0.0               0.0         0.0        0.0
614           frontend-0        trace.to_currencyservice-0.error_rate             30           1647770400         1647772140               0.0               0.0         0.0        0.0
662           frontend-1                   trace.from_root.error_rate             30           1647770400         1647772140               0.0               0.0         0.0        0.0
90         cartservice-1             trace.from_frontend-2.error_rate             30           1647770400         1647772140               0.0               0.0         0.0        0.0
93         cartservice-1                      trace.self.duration_p95             30           1647770400         1647772140               0.0               0.0         0.0        0.0
94         cartservice-1                        trace.self.error_rate             30           1647770400         1647772140               0.0               0.0         0.0        0.0
82         cartservice-1             trace.from_frontend-0.error_rate             30           1647770400         1647772140               0.0               0.0         0.0        0.0,                  cmdb_id                                   trace_name  anomaly_count  earliest_anomaly_ts  latest_anomaly_ts  window_min_value  window_max_value  global_p95  global_p5
14           adservice-1             trace.from_frontend-0.error_rate             30           1647770400         1647772140               0.0               0.0         0.0        0.0
1230  shippingservice2-0            trace.from_frontend2-0.error_rate             30           1647770400         1647772140               0.0               0.0         0.0        0.0
10           adservice-0             trace.from_frontend-2.error_rate             30           1647770400         1647772140               0.0               0.0         0.0        0.0
22           adservice-1             trace.from_frontend-2.error_rate             30           1647770400         1647772140               0.0               0.0         0.0        0.0
2            adservice-0             trace.from_frontend-0.error_rate             30           1647770400         1647772140               0.0               0.0         0.0        0.0
...                  ...                                          ...            ...                  ...                ...               ...               ...         ...        ...
714           frontend-1        trace.to_currencyservice-2.error_rate             30           1647770400         1647772140               0.0               0.0         0.0        0.0
718           frontend-1  trace.to_productcatalogservice-0.error_rate             30           1647770400         1647772140               0.0               0.0         0.0        0.0
746           frontend-1        trace.to_shippingservice-1.error_rate             30           1647770400         1647772140               0.0               0.0         0.0        0.0
750           frontend-1        trace.to_shippingservice-2.error_rate             30           1647770400         1647772140               0.0               0.0         0.0        0.0
738           frontend-1  trace.to_recommendationservice-2.error_rate             30           1647770400         1647772140               0.0               0.0         0.0        0.0

[50 rows x 9 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# Reuse df_metric and thresholds if available in kernel, otherwise load/recompute
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv', dtype={'timestamp': int})

df_metric['timestamp'] = df_metric['timestamp'].astype(int)

try:
    thresholds
except NameError:
    thresholds = df_metric.groupby(['cmdb_id', 'kpi_name'])['value'].agg(
        global_p95 = lambda x: float(np.percentile(x.values, 95)),
        global_p5  = lambda x: float(np.percentile(x.values, 5))
    ).reset_index()

# Targets
targets = ['checkoutservice-1', 'frontend-2']

# 1) Thresholds for target cmdb_ids
thr_targets = thresholds[thresholds['cmdb_id'].isin(targets)].copy()

# 2) Exclude constant/config series where global_p95 == global_p5
thr_variable = thr_targets[thr_targets['global_p95'] != thr_targets['global_p5']].copy()

# 3) Incident window (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
start_local = tz.localize(datetime(2022, 3, 20, 18, 0, 0))
end_local   = tz.localize(datetime(2022, 3, 20, 18, 30, 0))
start_ts = int(start_local.astimezone(pytz.utc).timestamp())
end_ts   = int(end_local.astimezone(pytz.utc).timestamp())

# All points in the window for targets
df_window_all = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts) & (df_metric['cmdb_id'].isin(targets))].copy()

# Keep only non-constant series by inner-joining thr_variable
df_window = df_window_all.merge(thr_variable, on=['cmdb_id','kpi_name'], how='inner')

# 4) Mark anomaly points: value >= global_p95 OR value <= global_p5
df_window['is_anom'] = (df_window['value'] >= df_window['global_p95']) | (df_window['value'] <= df_window['global_p5'])

# Anomaly points
df_anom_points = df_window[df_window['is_anom']].copy()

# Prepare result
result_cols = ['cmdb_id','kpi_name','anomaly_count','number_of_fault_runs',
               'earliest_fault_ts','latest_fault_ts','window_min_value','window_max_value',
               'global_p95','global_p5','severity_score']

if df_anom_points.empty:
    result_df = pd.DataFrame(columns=result_cols)
else:
    # Window-level min/max per KPI within the window (for the non-constant series)
    window_stats = df_window_all.merge(thr_variable, on=['cmdb_id','kpi_name'], how='inner').groupby(
        ['cmdb_id','kpi_name'])['value'].agg(window_min_value='min', window_max_value='max').reset_index()

    # Function to summarize consecutive runs where timestamps differ by exactly 60 seconds
    def summarize_runs(group):
        g = group.sort_values('timestamp')
        diffs = g['timestamp'].diff().fillna(999999)
        run_id = (diffs != 60).cumsum()
        runs = g.groupby(run_id)['timestamp'].agg(start_ts='min', end_ts='max', count='count').reset_index(drop=True)
        anomaly_count = int(g.shape[0])
        number_of_fault_runs = int(runs.shape[0])
        earliest_fault_ts = int(runs['start_ts'].min())
        latest_fault_ts = int(runs['end_ts'].max())
        return pd.Series({
            'anomaly_count': anomaly_count,
            'number_of_fault_runs': number_of_fault_runs,
            'earliest_fault_ts': earliest_fault_ts,
            'latest_fault_ts': latest_fault_ts
        })

    anom_summary = df_anom_points.groupby(['cmdb_id','kpi_name']).apply(summarize_runs).reset_index()

    # Merge with thresholds and window stats
    anom_agg = anom_summary.merge(thr_variable, on=['cmdb_id','kpi_name'], how='left') \
                           .merge(window_stats, on=['cmdb_id','kpi_name'], how='left')

    # Compute severity_score:
    # severity = max(window_max - global_p95 (if window_max > p95), global_p5 - window_min (if window_min < p5))
    anom_agg['window_min_value'] = pd.to_numeric(anom_agg['window_min_value'], errors='coerce')
    anom_agg['window_max_value'] = pd.to_numeric(anom_agg['window_max_value'], errors='coerce')

    def compute_severity(row):
        wmax = row['window_max_value']
        wmin = row['window_min_value']
        p95 = row['global_p95']
        p5  = row['global_p5']
        sev1 = (wmax - p95) if pd.notna(wmax) and (wmax > p95) else 0.0
        sev2 = (p5 - wmin) if pd.notna(wmin) and (wmin < p5) else 0.0
        return float(max(sev1, sev2))

    anom_agg['severity_score'] = anom_agg.apply(compute_severity, axis=1)

    # Ensure integer timestamps
    anom_agg['earliest_fault_ts'] = anom_agg['earliest_fault_ts'].astype(int)
    anom_agg['latest_fault_ts']   = anom_agg['latest_fault_ts'].astype(int)

    # Final compact table: ordered by severity desc, then anomaly_count desc, limit 20
    result_df = anom_agg[['cmdb_id','kpi_name','anomaly_count','number_of_fault_runs',
                         'earliest_fault_ts','latest_fault_ts','window_min_value','window_max_value',
                         'global_p95','global_p5','severity_score']].sort_values(
                             by=['severity_score','anomaly_count'], ascending=[False, False]
                         ).head(20)

# Display compact result (stored in result_df)
result_df

```
Out[5]:
```
Summary of metric anomalies for checkoutservice-1 and frontend-2 during March 20, 2022 18:00–18:30 (UTC+8):

What was done
- Computed global P95 and P5 per (cmdb_id, kpi_name) using the full history, excluded constant/config series (global_p95 == global_p5).
- Filtered points inside the incident window and flagged those >= global_p95 or <= global_p5.
- Grouped anomaly points into consecutive 60s runs and computed per-KPI aggregates including a severity_score (max distance beyond p95 or below p5).

Key findings (top results by severity)
- frontend-2 has the largest deviations and most severe series:
  - Top item: frontend-2, mesh.source.frontend.jaeger-collector... — anomaly_count=5, 3 fault runs, earliest/ latest fault ts span the window, window_min=448,875, window_max=698,850, global_p95=694,950, global_p5=549,075, severity_score=100,200 (large exceedance above p95).
  - Other high-severity frontend-2 mesh/istio series (destination/ source to various downstream services) show persistent anomalies (anomaly_count between 2–11, number_of_fault_runs 3–5) and substantial deviations (severity scores from ~49,650 down to a few hundred).
  - Many frontend-2 anomalies are mesh/istio request/response counters (traffic/collector metrics) showing window values well above their historical p95 in some cases.

- checkoutservice-1 also shows anomalies (lower severity than the top frontend-2 items):
  - Examples: mesh.source.checkoutservice.jaeger-collector... — anomaly_count=4, 3 runs, window_min=19,300, window_max=52,000, severity_score≈2,825.
  - mesh.source.checkoutservice.* calls (productcatalogservice, currencyservice, destination.frontend) show anomaly_count=4–5 with severity scores in the hundreds to low thousands.

Behavior pattern and interpretation
- These anomalies are mostly in mesh/istio call-volume or collector-related metrics (traffic/request/response counts) rather than low-level resource metrics.
- Many anomalies are persistent across the window (multiple runs spanning the incident), indicating sustained deviations from historical percentiles rather than single isolated spikes.
- severity_score measures how far values exceed p95 (or fall below p5); the highest severity items (frontend-2) show very large exceedances and should be prioritized.

Suggested next steps
- Inspect the raw time series for the top KPI rows to confirm whether the high values correspond to traffic spikes, configuration changes, or telemetry artifacts.
- Correlate with traces/logs (especially frontend→downstream call traces and jaeger collector behavior) to see if increased traffic or errors coincide.
- If confirmed as traffic/latency incidents, investigate load sources, rate-limiting, or upstream changes; if telemetry artifact, consider excluding collector/constant counters from this percentile rule or use stricter anomaly logic.

If you want, I can:
- Show the full intermediate tables for these two cmdb_ids,
- Plot the exact per-minute series (text summary) for any of the top KPIs,
- Or re-run detection using a stricter rule (strict >p95 / <p5) to reduce borderline flags.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  anomaly_count  number_of_fault_runs  earliest_fault_ts  latest_fault_ts  window_min_value  window_max_value  global_p95  global_p5  severity_score
113         frontend-2  mesh.source.frontend.jaeger-collector.istio_re...              5                     3         1647771000       1647772200         448875.00          698850.0   694950.00   549075.0       100200.00
87          frontend-2  mesh.destination.unknown.frontend.istio_respon...              5                     3         1647771420       1647772200         321775.00          509450.0   503600.00   371425.0        49650.00
117         frontend-2  mesh.source.frontend.productcatalogservice.ist...              7                     5         1647771000       1647772200         269700.00          435550.0   419275.00   310775.0        41075.00
107         frontend-2  mesh.source.frontend.currencyservice.istio_req...              6                     5         1647771120       1647772140         151125.00          286750.0   269700.00   185225.0        34100.00
121         frontend-2  mesh.source.frontend.productcatalogservice.ist...              5                     3         1647771000       1647772200          54755.00           88585.0    87350.00    68075.0        13320.00
95          frontend-2  mesh.source.frontend.cartservice.istio_request...              6                     4         1647771000       1647772200          50375.00           79825.0    79050.00    62000.0        11625.00
111         frontend-2  mesh.source.frontend.currencyservice.istio_res...              5                     5         1647770940       1647772140          26437.50           48600.0    45862.50    32887.5         6450.00
123         frontend-2  mesh.source.frontend.recommendationservice.ist...              7                     5         1647771000       1647772200          41850.00           68975.0    65100.00    47275.0         5425.00
89          frontend-2  mesh.source.frontend.adservice.istio_request_b...              2                     2         1647771000       1647772140          30700.00           57625.0    57625.00    34175.0         3475.00
40   checkoutservice-1  mesh.source.checkoutservice.jaeger-collector.i...              4                     3         1647770880       1647772200          19300.00           52000.0    49175.00    21075.0         2825.00
81          frontend-2  mesh.destination.unknown.frontend.istio_reques...              5                     3         1647771000       1647772200          10680.00           17497.5    17290.00    13450.0         2770.00
127         frontend-2  mesh.source.frontend.recommendationservice.ist...              7                     5         1647771000       1647772200           7425.00           12237.5    11550.00     8387.5          962.50
33   checkoutservice-1  mesh.source.checkoutservice.emailservice.istio...              5                     3         1647770880       1647772200           1850.00            7300.0     6475.00     2775.0          925.00
99          frontend-2  mesh.source.frontend.cartservice.istio_respons...              5                     3         1647771780       1647772200           5692.50            8820.0     8760.00     6540.0          847.50
116         frontend-2  mesh.source.frontend.jaeger-collector.istio_re...              5                     3         1647771000       1647772200           5223.75            7661.5     7512.25     6069.5          845.75
101         frontend-2  mesh.source.frontend.checkoutservice.istio_req...             11                     5         1647770700       1647772020            825.00            5775.0     4950.00     1650.0          825.00
50   checkoutservice-1  mesh.source.checkoutservice.productcatalogserv...              4                     3         1647770880       1647772200           4125.00           11550.0    10725.00     4950.0          825.00
26   checkoutservice-1  mesh.source.checkoutservice.currencyservice.is...              4                     3         1647770880       1647772200           6200.00           16275.0    15500.00     6975.0          775.00
18   checkoutservice-1  mesh.destination.frontend.checkoutservice.isti...              5                     3         1647770880       1647772200           1450.00            5800.0     5075.00     2175.0          725.00
93          frontend-2  mesh.source.frontend.adservice.istio_response_...              3                     3         1647770880       1647772140           6775.00           13537.5    12952.50     7160.0          585.00```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# Reuse existing df_trace and df_err if present, else load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv', dtype={'timestamp': int})

try:
    df_err
except NameError:
    # load error_logs.csv
    df_err = pd.read_csv('error_logs.csv', dtype={'timestamp': int})

# Ensure timestamp integers
df_trace['timestamp'] = df_trace['timestamp'].astype(int)
df_err['timestamp'] = df_err['timestamp'].astype(int)

# Targets
targets = ['frontend-2', 'checkoutservice-1']

# 1) Compute global P95 and P5 for each (cmdb_id, trace_name) across full series
trace_thresholds = df_trace.groupby(['cmdb_id', 'trace_name'])['value'].agg(
    global_p95 = lambda x: float(np.percentile(x.values, 95)),
    global_p5  = lambda x: float(np.percentile(x.values, 5))
).reset_index()

# Exclude constant series where p95 == p5
trace_thresholds_var = trace_thresholds[trace_thresholds['global_p95'] != trace_thresholds['global_p5']].copy()

# 2) Incident window (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
start_local = tz.localize(datetime(2022, 3, 20, 18, 0, 0))
end_local   = tz.localize(datetime(2022, 3, 20, 18, 30, 0))
start_ts = int(start_local.astimezone(pytz.utc).timestamp())
end_ts   = int(end_local.astimezone(pytz.utc).timestamp())

# Filter trace points to window and to target cmdb_ids
df_trace_window_all = df_trace[(df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts) & (df_trace['cmdb_id'].isin(targets))].copy()

# Keep only non-constant trace series by merging thresholds_var (inner join)
df_trace_window = df_trace_window_all.merge(trace_thresholds_var, on=['cmdb_id','trace_name'], how='inner')

# 3) Mark anomaly points in window: value >= global_p95 OR value <= global_p5
df_trace_window['is_anom'] = (df_trace_window['value'] >= df_trace_window['global_p95']) | (df_trace_window['value'] <= df_trace_window['global_p5'])

# Anomaly points
df_trace_anom_points = df_trace_window[df_trace_window['is_anom']].copy()

# 4) Group consecutive anomaly points into runs and compute per-trace aggregates
result_cols = ['cmdb_id','trace_name','anomaly_count','number_of_fault_runs',
               'earliest_fault_ts','latest_fault_ts','window_min_value','window_max_value',
               'global_p95','global_p5']

if df_trace_anom_points.empty:
    trace_anom_head = pd.DataFrame(columns=result_cols)
else:
    # Window-level min/max for each (cmdb_id, trace_name) among all window points (not only anomaly points)
    window_stats = df_trace_window_all.merge(trace_thresholds_var, on=['cmdb_id','trace_name'], how='inner').groupby(
        ['cmdb_id','trace_name'])['value'].agg(window_min_value='min', window_max_value='max').reset_index()

    # Function to compute runs for a group's anomaly points
    def summarize_runs(g):
        g = g.sort_values('timestamp').reset_index(drop=True)
        # diff in seconds
        diffs = g['timestamp'].diff().fillna(999999)
        run_id = (diffs != 60).cumsum()
        runs = g.groupby(run_id)['timestamp'].agg(start_ts='min', end_ts='max').reset_index(drop=True)
        anomaly_count = int(g.shape[0])
        number_of_fault_runs = int(runs.shape[0])
        earliest_fault_ts = int(runs['start_ts'].min())
        latest_fault_ts = int(runs['end_ts'].max())
        return pd.Series({
            'anomaly_count': anomaly_count,
            'number_of_fault_runs': number_of_fault_runs,
            'earliest_fault_ts': earliest_fault_ts,
            'latest_fault_ts': latest_fault_ts
        })

    anom_summary = df_trace_anom_points.groupby(['cmdb_id','trace_name']).apply(summarize_runs).reset_index()

    # Merge with thresholds and window stats
    trace_anom_agg = anom_summary.merge(trace_thresholds_var, on=['cmdb_id','trace_name'], how='left') \
                                 .merge(window_stats, on=['cmdb_id','trace_name'], how='left')

    # Ensure integer timestamps
    trace_anom_agg['earliest_fault_ts'] = trace_anom_agg['earliest_fault_ts'].astype(int)
    trace_anom_agg['latest_fault_ts']   = trace_anom_agg['latest_fault_ts'].astype(int)

    # Compact table: order by anomaly_count desc, limit 20
    trace_anom_head = trace_anom_agg[['cmdb_id','trace_name','anomaly_count','number_of_fault_runs',
                                      'earliest_fault_ts','latest_fault_ts','window_min_value','window_max_value',
                                      'global_p95','global_p5']].sort_values('anomaly_count', ascending=False).head(20)

# 4) From error_logs.csv: return rows for the two cmdb_ids (limit 20) with columns timestamp, cmdb_id, message
error_rows = df_err[df_err['cmdb_id'].isin(targets)][['timestamp','cmdb_id','message']].copy().head(20)

# Display the two compact outputs (trace anomalies table and error log rows)
trace_anom_head, error_rows

```
Out[6]:
```
Summary of findings for frontend-2 and checkoutservice-1 (March 20, 2022 18:00–18:30 UTC+8)

Trace anomalies (compact):
- Many call-quality traces for both components show sustained anomalies inside the window (points >= global P95 or <= global P5), grouped into multi-minute fault runs.
- checkoutservice-1 (high counts):
  - trace.from_frontend-1.row_count: anomaly_count = 29 (2 runs), earliest = 1647770400, latest = 1647772140 — window values 1–2 vs global p95=2 / p5=1.
  - Multiple checkoutservice→paymentservice and checkoutservice→emailservice row_count traces: anomaly_count ≈ 20–22, many short runs (several runs per KPI), window values mostly 1–3 vs p95/p5 showing these are at historical extremes.
  - Several other checkoutservice→service row_count traces also flagged (productcatalog, shipping, currency).
- frontend-2 (mix of counts and latency):
  - Several to_checkoutservice and to_cartservice row_count traces flagged (anomaly_count ≈ 15–19).
  - Duration p95 anomalies: e.g., trace.to_cartservice-1.duration_p95 (anomaly_count = 21) and other duration_mean/p95 traces — these show elevated latencies in the window (non-constant, multiple runs).
  - Many frontend-2 mesh/istio traces show repeated deviations across the window (earliest to latest timestamps span most of the 30-min window).

Error logs (for these two components):
- Two error-log rows at timestamp 1647772098:
  - frontend-2: "severity: error, message: request error"
  - checkoutservice-1: "severity: warning, message: failed to send ord..." (truncated in table)
- Both error log entries fall within the incident window and coincide temporally with the anomalous trace activity.

Interpretation and recommended next steps:
- The pattern: elevated/at-extreme row_counts across many inter-service traces plus duration_p95 anomalies on frontend-2, together with error/warning log lines at 1647772098, suggest a problem impacting the checkout flow (increased/abnormal request behavior and some request errors around that time).
- Prioritize investigation on:
  1. Trace spans around timestamp 1647772098 (frontend-2 → checkoutservice and checkoutservice → paymentservice) to see error propagation and root span causing failures.
  2. Payment/email downstream services and network/mesh latency — many checkout→payment/email traces are anomalous.
  3. Application logs on paymentservice and checkoutservice for failures to send orders or connection errors (the checkoutservice error log indicates a send failure).
- Actions to take: inspect full per-minute series and raw traces for top anomalous trace_names, correlate with logs on paymentservice and network/mesh telemetry, and check whether traffic surge, request routing, or downstream failures caused the observed anomalies.

If you want, I can:
- Show the detailed per-trace time series around 1647772098 for specific trace_names,
- Or extract full (not truncated) error log messages for more context.

The original code execution output of IPython Kernel is also provided below for reference:

(               cmdb_id                                  trace_name  anomaly_count  number_of_fault_runs  earliest_fault_ts  latest_fault_ts  window_min_value  window_max_value  global_p95  global_p5
4    checkoutservice-1             trace.from_frontend-1.row_count             29                     2         1647770400       1647772140          1.000000          2.000000    2.000000   1.000000
40   checkoutservice-1         trace.to_paymentservice-1.row_count             22                     9         1647770400       1647772140          1.000000          2.000000    2.000000   1.000000
43   checkoutservice-1         trace.to_paymentservice-2.row_count             22                     9         1647770400       1647772140          1.000000          2.000000    2.000000   1.000000
31   checkoutservice-1           trace.to_emailservice-1.row_count             21                    10         1647770400       1647772140          1.000000          2.000000    2.000000   1.000000
81          frontend-2         trace.to_cartservice-1.duration_p95             21                     8         1647770460       1647772080          0.000000          0.001500    0.001000   0.000000
34   checkoutservice-1           trace.to_emailservice-2.row_count             20                     9         1647770400       1647772140          1.000000          2.000000    2.000000   1.000000
28   checkoutservice-1           trace.to_emailservice-0.row_count             20                    10         1647770400       1647772140          1.000000          2.000000    2.000000   1.000000
37   checkoutservice-1         trace.to_paymentservice-0.row_count             20                    10         1647770460       1647772140          1.000000          2.000000    2.000000   1.000000
93          frontend-2        trace.to_checkoutservice-2.row_count             19                    10         1647770400       1647772020          1.000000          2.000000    1.100000   1.000000
90          frontend-2        trace.to_checkoutservice-1.row_count             19                    11         1647770400       1647772080          1.000000          2.000000    1.100000   1.000000
7    checkoutservice-1             trace.from_frontend-2.row_count             19                    11         1647770400       1647772080          1.000000          2.000000    1.100000   1.000000
16   checkoutservice-1            trace.to_cartservice-2.row_count             18                     6         1647770400       1647772080          1.000000          3.000000    3.000000   1.000000
52   checkoutservice-1  trace.to_productcatalogservice-2.row_count             17                     8         1647770460       1647772140          1.000000          3.000000    3.000000   1.000000
61   checkoutservice-1        trace.to_shippingservice-2.row_count             17                     6         1647770400       1647772020          1.000000          3.000000    3.000000   1.000000
78          frontend-2         trace.to_cartservice-0.duration_p95             16                     4         1647770640       1647772080          0.000000          0.001000    0.001000   0.000000
19   checkoutservice-1        trace.to_currencyservice-0.row_count             16                     8         1647770400       1647772020          1.000000          4.000000    4.000000   1.000000
55   checkoutservice-1        trace.to_shippingservice-0.row_count             15                     7         1647770520       1647772080          1.000000          4.000000    3.000000   1.000000
49   checkoutservice-1  trace.to_productcatalogservice-1.row_count             15                     7         1647770460       1647772080          1.000000          3.000000    3.000000   1.000000
126         frontend-2        trace.to_shippingservice-1.row_count             15                     4         1647771060       1647772140          1.000000          6.000000    5.000000   2.000000
71          frontend-2          trace.to_adservice-1.duration_mean             15                     7         1647770460       1647772020          0.000016          0.000027    0.000025   0.000016,     timestamp            cmdb_id                                            message
0  1647772098         frontend-2            severity: error, message: request error
1  1647772098  checkoutservice-1  severity: warning, message: failed to send ord...)```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# Reuse existing dataframes if present, otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv', dtype={'timestamp': int})

try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv', dtype={'timestamp': int})

try:
    df_log
except NameError:
    df_log = pd.read_csv('log.csv', dtype={'timestamp': int})

try:
    df_err
except NameError:
    df_err = pd.read_csv('error_logs.csv', dtype={'timestamp': int})

# Ensure timestamp ints
df_metric['timestamp'] = df_metric['timestamp'].astype(int)
df_trace['timestamp'] = df_trace['timestamp'].astype(int)
df_log['timestamp'] = df_log['timestamp'].astype(int)
df_err['timestamp'] = df_err['timestamp'].astype(int)

# Targets and incident window (UTC+8)
targets = ['frontend-2', 'checkoutservice-1']
tz = pytz.timezone('Asia/Shanghai')
start_local = tz.localize(datetime(2022, 3, 20, 18, 0, 0))
end_local   = tz.localize(datetime(2022, 3, 20, 18, 30, 0))
start_ts = int(start_local.astimezone(pytz.utc).timestamp())
end_ts   = int(end_local.astimezone(pytz.utc).timestamp())

# -------------------------
# 1) Metrics summary
# -------------------------
# Compute global thresholds (reuse if available)
try:
    thresholds
except NameError:
    thresholds = df_metric.groupby(['cmdb_id','kpi_name'])['value'].agg(
        global_p95 = lambda x: float(np.percentile(x.values, 95)),
        global_p5  = lambda x: float(np.percentile(x.values, 5))
    ).reset_index()

# Keep only target cmdb_ids and non-constant series
thr_targets = thresholds[thresholds['cmdb_id'].isin(targets)].copy()
thr_var = thr_targets[thr_targets['global_p95'] != thr_targets['global_p5']].copy()

# Metric points in window for targets
df_metric_window_all = df_metric[(df_metric['timestamp'] >= start_ts) &
                                 (df_metric['timestamp'] <= end_ts) &
                                 (df_metric['cmdb_id'].isin(targets))].copy()

# Merge to keep only non-constant series
df_metric_win = df_metric_window_all.merge(thr_var, on=['cmdb_id','kpi_name'], how='inner')

# Mark anomalies
df_metric_win['is_anom'] = (df_metric_win['value'] >= df_metric_win['global_p95']) | (df_metric_win['value'] <= df_metric_win['global_p5'])

# Anomaly points
df_metric_anom = df_metric_win[df_metric_win['is_anom']].copy()

# Compute per-KPI anomaly summary (for severity calc)
if not df_metric_anom.empty:
    # anomaly counts and earliest/latest per (cmdb,kpi)
    metric_anom_kpi = df_metric_anom.groupby(['cmdb_id','kpi_name']).agg(
        anomaly_count = ('value','count'),
        earliest_anom_ts = ('timestamp','min'),
        latest_anom_ts = ('timestamp','max')
    ).reset_index()

    # window-level min/max for each KPI (from all points in window for non-constant series)
    window_stats = df_metric_win.groupby(['cmdb_id','kpi_name'])['value'].agg(
        window_min_value='min',
        window_max_value='max'
    ).reset_index()

    # Merge thresholds and window stats
    metric_anom_kpi = metric_anom_kpi.merge(thr_var, on=['cmdb_id','kpi_name'], how='left') \
                                     .merge(window_stats, on=['cmdb_id','kpi_name'], how='left')

    # Compute severity per KPI
    def metric_severity(row):
        wmax = row['window_max_value']
        wmin = row['window_min_value']
        p95 = row['global_p95']
        p5  = row['global_p5']
        sev_up = (wmax - p95) if pd.notna(wmax) and (wmax > p95) else 0.0
        sev_dn = (p5 - wmin) if pd.notna(wmin) and (wmin < p5) else 0.0
        return float(max(sev_up, sev_dn))

    metric_anom_kpi['severity'] = metric_anom_kpi.apply(metric_severity, axis=1)

else:
    metric_anom_kpi = pd.DataFrame(columns=['cmdb_id','kpi_name','anomaly_count','earliest_anom_ts','latest_anom_ts','global_p95','global_p5','window_min_value','window_max_value','severity'])

# Aggregate per cmdb_id: earliest_metric_anomaly_ts, number_of_metric_series_with_anomalies, max_metric_severity
metrics_summary_rows = []
for cm in targets:
    sub = metric_anom_kpi[metric_anom_kpi['cmdb_id']==cm]
    if sub.empty:
        metrics_summary_rows.append({
            'cmdb_id': cm,
            'earliest_metric_anomaly_ts': np.nan,
            'number_of_metric_series_with_anomalies': 0,
            'max_metric_severity': 0.0
        })
    else:
        earliest_ts_metric = int(sub['earliest_anom_ts'].min())
        n_series = int(sub.shape[0])
        max_sev = float(sub['severity'].max())
        metrics_summary_rows.append({
            'cmdb_id': cm,
            'earliest_metric_anomaly_ts': earliest_ts_metric,
            'number_of_metric_series_with_anomalies': n_series,
            'max_metric_severity': max_sev
        })

metrics_summary = pd.DataFrame(metrics_summary_rows)[['cmdb_id','earliest_metric_anomaly_ts','number_of_metric_series_with_anomalies','max_metric_severity']]

# -------------------------
# 2) Traces summary
# -------------------------
# Compute trace thresholds (reuse if available)
try:
    trace_thresholds
except NameError:
    trace_thresholds = df_trace.groupby(['cmdb_id','trace_name'])['value'].agg(
        global_p95 = lambda x: float(np.percentile(x.values, 95)),
        global_p5  = lambda x: float(np.percentile(x.values, 5))
    ).reset_index()

# Keep only variable (non-constant) traces for targets
trace_thr_targets = trace_thresholds[trace_thresholds['cmdb_id'].isin(targets)].copy()
trace_thr_var = trace_thr_targets[trace_thr_targets['global_p95'] != trace_thr_targets['global_p5']].copy()

# Trace points in window for targets
df_trace_window_all = df_trace[(df_trace['timestamp'] >= start_ts) &
                               (df_trace['timestamp'] <= end_ts) &
                               (df_trace['cmdb_id'].isin(targets))].copy()

# Merge to keep only non-constant traces
df_trace_win = df_trace_window_all.merge(trace_thr_var, on=['cmdb_id','trace_name'], how='inner')

# Mark anomalies
df_trace_win['is_anom'] = (df_trace_win['value'] >= df_trace_win['global_p95']) | (df_trace_win['value'] <= df_trace_win['global_p5'])

df_trace_anom = df_trace_win[df_trace_win['is_anom']].copy()

# Per-trace anomaly aggregation (earliest/latest)
if not df_trace_anom.empty:
    trace_anom_trace = df_trace_anom.groupby(['cmdb_id','trace_name']).agg(
        anomaly_count = ('value','count'),
        earliest_anom_ts = ('timestamp','min'),
        latest_anom_ts = ('timestamp','max')
    ).reset_index()
else:
    trace_anom_trace = pd.DataFrame(columns=['cmdb_id','trace_name','anomaly_count','earliest_anom_ts','latest_anom_ts'])

# Aggregate per cmdb_id for traces summary
trace_summary_rows = []
for cm in targets:
    sub = trace_anom_trace[trace_anom_trace['cmdb_id']==cm]
    if sub.empty:
        trace_summary_rows.append({
            'cmdb_id': cm,
            'earliest_trace_anomaly_ts': np.nan,
            'number_of_trace_series_with_anomalies': 0
        })
    else:
        earliest_ts_trace = int(sub['earliest_anom_ts'].min())
        n_trace_series = int(sub.shape[0])
        trace_summary_rows.append({
            'cmdb_id': cm,
            'earliest_trace_anomaly_ts': earliest_ts_trace,
            'number_of_trace_series_with_anomalies': n_trace_series
        })

traces_summary = pd.DataFrame(trace_summary_rows)[['cmdb_id','earliest_trace_anomaly_ts','number_of_trace_series_with_anomalies']]

# -------------------------
# 3) Logs / error_logs outputs
# -------------------------
# Error logs for targets (show timestamp, cmdb_id, message). Keep compact: show up to 20 rows
error_rows = df_err[df_err['cmdb_id'].isin(targets)][['timestamp','cmdb_id','message']].copy().head(20)

# log.csv rows for targets in window, ordered by timestamp asc, limit 50, columns: timestamp, cmdb_id, log_name, value
log_window = df_log[(df_log['timestamp'] >= start_ts) &
                    (df_log['timestamp'] <= end_ts) &
                    (df_log['cmdb_id'].isin(targets))][['timestamp','cmdb_id','log_name','value']].copy()

log_window_sorted = log_window.sort_values('timestamp', ascending=True).head(50)

# Final compact outputs (stored in variables): metrics_summary (<=2 rows), traces_summary (<=2 rows), error_rows (<=20), log_window_sorted (<=50)
metrics_summary, traces_summary, error_rows, log_window_sorted

```
Out[7]:
```
Summary (focused on frontend-2 and checkoutservice-1, March 20, 2022 18:00–18:30 UTC+8)

Metrics (metric.csv)
- Earliest metric anomaly for both components: 1647770400 (start of the window).
- Number of distinct metric series with anomalies in the window:
  - frontend-2: 67 series
  - checkoutservice-1: 62 series
- Max metric severity (largest distance beyond global percentile among the KPI anomalies):
  - frontend-2: 100,200.0 (very large deviation on at least one KPI)
  - checkoutservice-1: 2,825.0 (noticeable but much smaller than frontend-2)
- Interpretation: frontend-2 has many metric series anomalous and at least one with a very large exceedance vs historical P95; checkoutservice-1 also has many anomalous series but with lower peak severity.

Traces (trace.csv)
- Earliest trace anomaly for both components: 1647770400.
- Number of distinct trace-series with anomalies in window:
  - frontend-2: 68 series
  - checkoutservice-1: 62 series
- Interpretation: both components show broad trace-level deviations across many trace names during the window.

Logs
- error_logs.csv (both components, within window):
  - Two entries at timestamp 1647772098:
    - frontend-2: "severity: error, message: request error"
    - checkoutservice-1: "severity: warning, message: failed to send ord..." (truncated)
  - These error/warning entries occur near the same time and lie inside the incident window.
- log.csv (points for these two components in the window; shown up to 50 rows)
  - Many per-minute log.row_count entries (e.g., frontend-2 has large row_count values such as ~822 at 1647770400; checkoutservice-1 row_count examples like 37 at same timestamp).
  - log.error_count values are mostly 0 in the printed rows.
  - Full list of windowed log rows (up to 50) was produced and stored.

Overall conclusion & recommended next steps
- Both frontend-2 and checkoutservice-1 show broad anomalies in metrics and traces starting at the beginning of the window; frontend-2 has much higher metric severity on some KPIs.
- Error/warning log entries at 1647772098 for both services coincide with anomalous trace/metric activity — prioritize investigation around that timestamp.
- Recommended actions:
  1. Inspect trace spans and logs near 1647772098 (frontend-2 → checkoutservice, checkoutservice → payment/email) to locate error propagation and cause.
  2. Drill into the top metric KPI(s) for frontend-2 with the high severity to understand whether this is traffic surge, resource exhaustion, or telemetry artifact.
  3. Check downstream services (payment/email) for related errors.
  4. Correlate full per-minute series and raw logs/traces for the top anomalous trace_names and KPI names to confirm root cause.

The original code execution output of IPython Kernel is also provided below for reference:

(             cmdb_id  earliest_metric_anomaly_ts  number_of_metric_series_with_anomalies  max_metric_severity
0         frontend-2                  1647770400                                      67             100200.0
1  checkoutservice-1                  1647770400                                      62               2825.0,              cmdb_id  earliest_trace_anomaly_ts  number_of_trace_series_with_anomalies
0         frontend-2                 1647770400                                     68
1  checkoutservice-1                 1647770400                                     62,     timestamp            cmdb_id                                            message
0  1647772098         frontend-2            severity: error, message: request error
1  1647772098  checkoutservice-1  severity: warning, message: failed to send ord...,       timestamp            cmdb_id         log_name  value
14   1647770400  checkoutservice-1  log.error_count    0.0
15   1647770400  checkoutservice-1    log.row_count   37.0
34   1647770400         frontend-2  log.error_count    0.0
35   1647770400         frontend-2    log.row_count  822.0
74   1647770460  checkoutservice-1  log.error_count    0.0
..          ...                ...              ...    ...
681  1647771060  checkoutservice-1    log.row_count   56.0
700  1647771060         frontend-2  log.error_count    0.0
701  1647771060         frontend-2    log.row_count  791.0
740  1647771120  checkoutservice-1  log.error_count    0.0
741  1647771120  checkoutservice-1    log.row_count   34.0

[50 rows x 4 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# Reuse existing dataframes if present, else load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv', dtype={'timestamp': int})
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv', dtype={'timestamp': int})

# Ensure timestamp ints
df_metric['timestamp'] = df_metric['timestamp'].astype(int)
df_trace['timestamp'] = df_trace['timestamp'].astype(int)

# 1) Compute global thresholds for metric and trace (use existing vars if present)
try:
    thresholds  # metric thresholds
except NameError:
    thresholds = df_metric.groupby(['cmdb_id','kpi_name'])['value'].agg(
        global_p95 = lambda x: float(np.percentile(x.values, 95)),
        global_p5  = lambda x: float(np.percentile(x.values, 5))
    ).reset_index()

try:
    trace_thresholds
except NameError:
    trace_thresholds = df_trace.groupby(['cmdb_id','trace_name'])['value'].agg(
        global_p95 = lambda x: float(np.percentile(x.values, 95)),
        global_p5  = lambda x: float(np.percentile(x.values, 5))
    ).reset_index()

# 2) Exclude constant/config series where p95 == p5
thr_var_metric = thresholds[thresholds['global_p95'] != thresholds['global_p5']].copy()
thr_var_trace  = trace_thresholds[trace_thresholds['global_p95'] != trace_thresholds['global_p5']].copy()

# 3) Incident window (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
start_local = tz.localize(datetime(2022, 3, 20, 18, 0, 0))
end_local   = tz.localize(datetime(2022, 3, 20, 18, 30, 0))
start_ts = int(start_local.astimezone(pytz.utc).timestamp())
end_ts   = int(end_local.astimezone(pytz.utc).timestamp())

# Filter metric and trace points to the window
df_metric_window_all = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()
df_trace_window_all  = df_trace[(df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts)].copy()

# Merge thresholds (inner join) to keep only non-constant series, then mark anomalies
df_metric_win = df_metric_window_all.merge(thr_var_metric, on=['cmdb_id','kpi_name'], how='inner')
df_metric_win['is_anom'] = (df_metric_win['value'] >= df_metric_win['global_p95']) | (df_metric_win['value'] <= df_metric_win['global_p5'])
df_metric_anom = df_metric_win[df_metric_win['is_anom']].copy()   # store full intermediate

df_trace_win = df_trace_window_all.merge(thr_var_trace, on=['cmdb_id','trace_name'], how='inner')
df_trace_win['is_anom'] = (df_trace_win['value'] >= df_trace_win['global_p95']) | (df_trace_win['value'] <= df_trace_win['global_p5'])
df_trace_anom = df_trace_win[df_trace_win['is_anom']].copy()     # store full intermediate

# 4) Aggregate anomaly points per timestamp across both metrics and traces
# Metric anomaly counts per timestamp
metric_counts_ts = df_metric_anom.groupby('timestamp').size().rename('metric_anomaly_count').reset_index()
# Trace anomaly counts per timestamp
trace_counts_ts  = df_trace_anom.groupby('timestamp').size().rename('trace_anomaly_count').reset_index()

# Merge counts and compute total anomaly_count
anom_ts = pd.merge(metric_counts_ts, trace_counts_ts, on='timestamp', how='outer').fillna(0)
anom_ts['metric_anomaly_count'] = anom_ts['metric_anomaly_count'].astype(int)
anom_ts['trace_anomaly_count']  = anom_ts['trace_anomaly_count'].astype(int)
anom_ts['anomaly_count'] = anom_ts['metric_anomaly_count'] + anom_ts['trace_anomaly_count']

# Ensure integer timestamp
anom_ts['timestamp'] = anom_ts['timestamp'].astype(int)

# 5A) Top 10 timestamps ordered by anomaly_count desc
top10_ts = anom_ts.sort_values(['anomaly_count','timestamp'], ascending=[False, True]).head(10)[['timestamp','anomaly_count','metric_anomaly_count','trace_anomaly_count']].reset_index(drop=True)

# 5B) For each timestamp from A), compute contributions from frontend-2 and checkoutservice-1 (combined metrics+traces)
targets = ['frontend-2','checkoutservice-1']

# Combined anomaly points with cmdb_id and timestamp (metrics + traces)
# keep only needed columns to save memory
m = df_metric_anom[['timestamp','cmdb_id']].copy()
t = df_trace_anom[['timestamp','cmdb_id']].copy()
combined_anoms = pd.concat([m,t], ignore_index=True)

# Count per timestamp and cmdb_id
per_ts_cm = combined_anoms.groupby(['timestamp','cmdb_id']).size().rename('count').reset_index()

# Pivot to have columns for the two targets
pivot_targets = per_ts_cm[per_ts_cm['cmdb_id'].isin(targets)].pivot_table(index='timestamp', columns='cmdb_id', values='count', aggfunc='sum', fill_value=0).reset_index()

# Ensure both target columns exist
for col in targets:
    if col not in pivot_targets.columns:
        pivot_targets[col] = 0

# Rename columns to required names
pivot_targets = pivot_targets.rename(columns={'frontend-2':'frontend-2_anomaly_count', 'checkoutservice-1':'checkoutservice-1_anomaly_count'})

# Merge pivot info into top10_ts (left join on timestamp)
top10_with_targets = top10_ts.merge(pivot_targets, on='timestamp', how='left').fillna(0)
# Ensure integer counts
count_cols = ['frontend-2_anomaly_count','checkoutservice-1_anomaly_count']
for c in count_cols:
    top10_with_targets[c] = top10_with_targets[c].astype(int)

# Final compact outputs to display: top10_ts and top10_with_targets (both small)
top10_ts, top10_with_targets

```
Out[8]:
```
Summary of combined metric+trace anomalies (top 10 minutes within the incident window):

Top minutes by total anomaly count (timestamp, total, metrics vs traces)
- 1647772080 — anomaly_count 921 (metrics 684, traces 237)
- 1647772140 — anomaly_count 917 (metrics 694, traces 223)
- 1647772020 — anomaly_count 915 (metrics 672, traces 243)
- 1647771780 — anomaly_count 889 (metrics 644, traces 245)
- 1647771600 — anomaly_count 801 (metrics 581, traces 220)
- 1647770760 — anomaly_count 792 (metrics 585, traces 207)
- 1647771300 — anomaly_count 749 (metrics 548, traces 201)
- 1647772200 — anomaly_count 734 (metrics 734, traces 0)
- 1647770640 — anomaly_count 730 (metrics 514, traces 216)
- 1647771900 — anomaly_count 724 (metrics 504, traces 220)

Per-minute contribution from frontend-2 and checkoutservice-1 (for the above top minutes)
- 1647772080 — checkoutservice-1: 63, frontend-2: 56
- 1647772140 — checkoutservice-1: 64, frontend-2: 77
- 1647772020 — checkoutservice-1: 35, frontend-2: 52
- 1647771780 — checkoutservice-1: 27, frontend-2: 61
- 1647771600 — checkoutservice-1: 16, frontend-2: 16
- 1647770760 — checkoutservice-1: 39, frontend-2: 28
- 1647771300 — checkoutservice-1: 36, frontend-2: 22
- 1647772200 — checkoutservice-1: 54, frontend-2: 39
- 1647770640 — checkoutservice-1: 25, frontend-2: 21
- 1647771900 — checkoutservice-1: 22, frontend-2: 41

Interpretation / takeaways
- Anomaly volume is very large at these minutes (hundreds of series flagged), and metric anomalies dominate over trace anomalies in every top minute.
- frontend-2 and checkoutservice-1 each contribute tens of anomaly points per minute at the busiest minutes (e.g., ~50–80 anomalies each at 1647772080–2140), indicating both services are meaningfully involved in the anomalous activity.
- Minutes around 1647772080–2140 are the most severe and should be prioritized for investigation (correlate traces, metrics, and logs at those timestamps).

Recommended next steps
- Inspect full raw traces and metric KPI details at timestamps 1647772080 and 1647772140 to find root causes.
- Correlate with error log entries and application logs around those minutes for frontend-2 and checkoutservice-1.
- If needed, filter out constant/config series or refine anomaly thresholds to reduce noise and focus on actionable deviations.

The original code execution output of IPython Kernel is also provided below for reference:

(    timestamp  anomaly_count  metric_anomaly_count  trace_anomaly_count
0  1647772080            921                   684                  237
1  1647772140            917                   694                  223
2  1647772020            915                   672                  243
3  1647771780            889                   644                  245
4  1647771600            801                   581                  220
5  1647770760            792                   585                  207
6  1647771300            749                   548                  201
7  1647772200            734                   734                    0
8  1647770640            730                   514                  216
9  1647771900            724                   504                  220,     timestamp  anomaly_count  metric_anomaly_count  trace_anomaly_count  checkoutservice-1_anomaly_count  frontend-2_anomaly_count
0  1647772080            921                   684                  237                               63                        56
1  1647772140            917                   694                  223                               64                        77
2  1647772020            915                   672                  243                               35                        52
3  1647771780            889                   644                  245                               27                        61
4  1647771600            801                   581                  220                               16                        16
5  1647770760            792                   585                  207                               39                        28
6  1647771300            749                   548                  201                               36                        22
7  1647772200            734                   734                    0                               54                        39
8  1647770640            730                   514                  216                               25                        21
9  1647771900            724                   504                  220                               22                        41)```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# Reuse existing dataframes if available, else load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv', dtype={'timestamp': int})
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv', dtype={'timestamp': int})
try:
    df_err
except NameError:
    df_err = pd.read_csv('error_logs.csv', dtype={'timestamp': int})

# Ensure timestamp ints
df_metric['timestamp'] = df_metric['timestamp'].astype(int)
df_trace['timestamp'] = df_trace['timestamp'].astype(int)
df_err['timestamp'] = df_err['timestamp'].astype(int)

# Targets and incident window (UTC+8)
targets = ['frontend-2', 'checkoutservice-1']
tz = pytz.timezone('Asia/Shanghai')
start_local = tz.localize(datetime(2022, 3, 20, 18, 0, 0))
end_local   = tz.localize(datetime(2022, 3, 20, 18, 30, 0))
start_ts = int(start_local.astimezone(pytz.utc).timestamp())
end_ts   = int(end_local.astimezone(pytz.utc).timestamp())

# 1) Compute global thresholds (full series) for metric and trace if not already computed
try:
    thresholds
except NameError:
    thresholds = df_metric.groupby(['cmdb_id','kpi_name'])['value'].agg(
        global_p95 = lambda x: float(np.percentile(x.values, 95)),
        global_p5  = lambda x: float(np.percentile(x.values, 5))
    ).reset_index()

try:
    trace_thresholds
except NameError:
    trace_thresholds = df_trace.groupby(['cmdb_id','trace_name'])['value'].agg(
        global_p95 = lambda x: float(np.percentile(x.values, 95)),
        global_p5  = lambda x: float(np.percentile(x.values, 5))
    ).reset_index()

# 2) Exclude constant/config series where p95 == p5
thr_var_metric = thresholds[thresholds['global_p95'] != thresholds['global_p5']].copy()
thr_var_trace  = trace_thresholds[trace_thresholds['global_p95'] != trace_thresholds['global_p5']].copy()

# 3) Filter points to window and mark anomalies (value >= p95 OR value <= p5) after merging thresholds (inner join)
# Metric window and anomalies (keep non-constant series only)
df_metric_window_all = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()
df_metric_win = df_metric_window_all.merge(thr_var_metric, on=['cmdb_id','kpi_name'], how='inner')
df_metric_win['is_anom'] = (df_metric_win['value'] >= df_metric_win['global_p95']) | (df_metric_win['value'] <= df_metric_win['global_p5'])
df_metric_anom = df_metric_win[df_metric_win['is_anom']].copy()   # full intermediate

# Trace window and anomalies (keep non-constant series only)
df_trace_window_all = df_trace[(df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts)].copy()
df_trace_win = df_trace_window_all.merge(thr_var_trace, on=['cmdb_id','trace_name'], how='inner')
df_trace_win['is_anom'] = (df_trace_win['value'] >= df_trace_win['global_p95']) | (df_trace_win['value'] <= df_trace_win['global_p5'])
df_trace_anom = df_trace_win[df_trace_win['is_anom']].copy()     # full intermediate

# 4) Aggregate anomalies per minute for each component separately
# Metric anomaly counts per (timestamp, cmdb_id)
metric_counts = df_metric_anom.groupby(['timestamp','cmdb_id']).size().rename('metric_anomaly_count').reset_index()

# Trace anomaly counts per (timestamp, cmdb_id)
trace_counts = df_trace_anom.groupby(['timestamp','cmdb_id']).size().rename('trace_anomaly_count').reset_index()

# Create full set of timestamps x cmdb_id combinations present in either anomalies to ensure zeros are explicit
combined_ts_cm = pd.merge(metric_counts, trace_counts, on=['timestamp','cmdb_id'], how='outer').fillna(0)
combined_ts_cm['metric_anomaly_count'] = combined_ts_cm['metric_anomaly_count'].astype(int)
combined_ts_cm['trace_anomaly_count']  = combined_ts_cm['trace_anomaly_count'].astype(int)
combined_ts_cm['total_anomaly_count']  = combined_ts_cm['metric_anomaly_count'] + combined_ts_cm['trace_anomaly_count']
combined_ts_cm['timestamp'] = combined_ts_cm['timestamp'].astype(int)

# Keep only target components
combined_targets = combined_ts_cm[combined_ts_cm['cmdb_id'].isin(targets)].copy()

# For completeness, if a target has no anomalies at some timestamp but the other does, ensure zeros are present:
# Create all timestamps in window (per-minute) present in either metric or trace anomalies to align selection
all_timestamps = pd.Index(sorted(combined_targets['timestamp'].unique()))

# 5) For each component return top 5 timestamps ordered by total_anomaly_count desc
top_list = []
for cm in targets:
    sub = combined_targets[combined_targets['cmdb_id']==cm].copy()
    if sub.empty:
        continue
    top5 = sub.sort_values(['total_anomaly_count','timestamp'], ascending=[False, True]).head(5)
    # select required columns
    top5_sel = top5[['cmdb_id','timestamp','total_anomaly_count','metric_anomaly_count','trace_anomaly_count']].copy()
    top_list.append(top5_sel)

# Concatenate top results (will be <=10 rows)
if top_list:
    top5_per_component = pd.concat(top_list, ignore_index=True)
else:
    top5_per_component = pd.DataFrame(columns=['cmdb_id','timestamp','total_anomaly_count','metric_anomaly_count','trace_anomaly_count'])

# 6) Error logs for the two cmdb_ids (full rows limited to 20)
error_rows = df_err[df_err['cmdb_id'].isin(targets)][['timestamp','cmdb_id','message']].copy().head(20)
error_rows['timestamp'] = error_rows['timestamp'].astype(int)

# Final compact outputs to display: top5_per_component (<=10 rows) and error_rows (<=20 rows)
top5_per_component, error_rows

```
Out[9]:
```
Summary for frontend-2 and checkoutservice-1 (metrics + traces, March 20, 2022 18:00–18:30 UTC+8)

Top anomalous minutes per component (total_anomaly_count, metric_anomaly_count, trace_anomaly_count)
- frontend-2
  1. 1647772140 — total 77 (metrics 54, traces 23)
  2. 1647771780 — total 61 (metrics 36, traces 25)
  3. 1647772080 — total 56 (metrics 33, traces 23)
  4. 1647772020 — total 52 (metrics 25, traces 27)
  5. 1647771000 — total 44 (metrics 34, traces 10)

- checkoutservice-1
  1. 1647772140 — total 64 (metrics 53, traces 11)
  2. 1647772080 — total 63 (metrics 30, traces 33)
  3. 1647772200 — total 54 (metrics 54, traces 0)
  4. 1647771720 — total 44 (metrics 24, traces 20)
  5. 1647771660 — total 42 (metrics 25, traces 17)

Error log rows (within the window) for these components
- 1647772098 — frontend-2 — "severity: error, message: request error"
- 1647772098 — checkoutservice-1 — "severity: warning, message: failed to send ord..." (truncated)

Key takeaways
- Both services exhibit concentrated anomaly activity at the same peak minutes (notably around 1647772140 and 1647772080).
- frontend-2 anomalies are often metric-dominant at some minutes (e.g., 1647772140), while checkoutservice-1 shows both metric and trace contributions (e.g., 1647772080 has many trace anomalies).
- Error/warning log entries for both components occur at 1647772098, which lies between the top anomalous minutes and should be investigated as a potential symptom or timestamp of interest.

Suggested next steps
- Inspect raw traces and top KPI series at 1647772080–2140 to identify root cause and direction of failures.
- Correlate trace spans around 1647772098 to follow error propagation between frontend-2 and checkoutservice-1.
- Check downstream services (payment/email) and network/mesh telemetry for coincident anomalies.

The original code execution output of IPython Kernel is also provided below for reference:

(             cmdb_id   timestamp  total_anomaly_count  metric_anomaly_count  trace_anomaly_count
0         frontend-2  1647772140                   77                    54                   23
1         frontend-2  1647771780                   61                    36                   25
2         frontend-2  1647772080                   56                    33                   23
3         frontend-2  1647772020                   52                    25                   27
4         frontend-2  1647771000                   44                    34                   10
5  checkoutservice-1  1647772140                   64                    53                   11
6  checkoutservice-1  1647772080                   63                    30                   33
7  checkoutservice-1  1647772200                   54                    54                    0
8  checkoutservice-1  1647771720                   44                    24                   20
9  checkoutservice-1  1647771660                   42                    25                   17,     timestamp            cmdb_id                                            message
0  1647772098         frontend-2            severity: error, message: request error
1  1647772098  checkoutservice-1  severity: warning, message: failed to send ord...)```
```

In [ ]:
# In[10]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# Reuse existing DataFrames if present; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv', dtype={'timestamp': int})
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv', dtype={'timestamp': int})
try:
    df_log
except NameError:
    df_log = pd.read_csv('log.csv', dtype={'timestamp': int})
try:
    df_err
except NameError:
    df_err = pd.read_csv('error_logs.csv', dtype={'timestamp': int})

# Ensure integer timestamps
df_metric['timestamp'] = df_metric['timestamp'].astype(int)
df_trace['timestamp'] = df_trace['timestamp'].astype(int)
df_log['timestamp'] = df_log['timestamp'].astype(int)
df_err['timestamp'] = df_err['timestamp'].astype(int)

# Identify paymentservice cmdb_ids present (prefix 'paymentservice')
payment_cmdbs_metric = df_metric[df_metric['cmdb_id'].str.startswith('paymentservice')]['cmdb_id'].unique()
payment_cmdbs_trace  = df_trace[df_trace['cmdb_id'].str.startswith('paymentservice')]['cmdb_id'].unique()
payment_cmdbs_log    = df_log[df_log['cmdb_id'].str.startswith('paymentservice')]['cmdb_id'].unique()
payment_cmdbs_err    = df_err[df_err['cmdb_id'].str.startswith('paymentservice')]['cmdb_id'].unique()

payment_cmdbs = pd.Index(np.unique(np.concatenate([payment_cmdbs_metric, payment_cmdbs_trace, payment_cmdbs_log, payment_cmdbs_err]))).tolist()

# If none found, we'll still proceed with empty results
# Incident window (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
start_local = tz.localize(datetime(2022, 3, 20, 18, 0, 0))
end_local   = tz.localize(datetime(2022, 3, 20, 18, 30, 0))
start_ts = int(start_local.astimezone(pytz.utc).timestamp())
end_ts   = int(end_local.astimezone(pytz.utc).timestamp())

# 1) Compute global P95 and P5 per series for payment cmdbs using full series (do NOT filter by window)
if len(payment_cmdbs) > 0:
    metric_payment_full = df_metric[df_metric['cmdb_id'].isin(payment_cmdbs)].copy()
    trace_payment_full  = df_trace[df_trace['cmdb_id'].isin(payment_cmdbs)].copy()

    metric_thr_payment = metric_payment_full.groupby(['cmdb_id','kpi_name'])['value'].agg(
        global_p95 = lambda x: float(np.percentile(x.values, 95)),
        global_p5  = lambda x: float(np.percentile(x.values, 5))
    ).reset_index()

    trace_thr_payment = trace_payment_full.groupby(['cmdb_id','trace_name'])['value'].agg(
        global_p95 = lambda x: float(np.percentile(x.values, 95)),
        global_p5  = lambda x: float(np.percentile(x.values, 5))
    ).reset_index()

    # 2) Exclude constant/config series where p95 == p5
    metric_thr_var = metric_thr_payment[metric_thr_payment['global_p95'] != metric_thr_payment['global_p5']].copy()
    trace_thr_var  = trace_thr_payment[trace_thr_payment['global_p95'] != trace_thr_payment['global_p5']].copy()

    # 3) Filter points to the incident window and mark anomalies
    df_metric_window = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts) & (df_metric['cmdb_id'].isin(payment_cmdbs))].copy()
    df_metric_win = df_metric_window.merge(metric_thr_var, on=['cmdb_id','kpi_name'], how='inner')
    df_metric_win['is_anom'] = (df_metric_win['value'] >= df_metric_win['global_p95']) | (df_metric_win['value'] <= df_metric_win['global_p5'])
    df_metric_anom = df_metric_win[df_metric_win['is_anom']].copy()

    df_trace_window = df_trace[(df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts) & (df_trace['cmdb_id'].isin(payment_cmdbs))].copy()
    df_trace_win = df_trace_window.merge(trace_thr_var, on=['cmdb_id','trace_name'], how='inner')
    df_trace_win['is_anom'] = (df_trace_win['value'] >= df_trace_win['global_p95']) | (df_trace_win['value'] <= df_trace_win['global_p5'])
    df_trace_anom = df_trace_win[df_trace_win['is_anom']].copy()
else:
    # empty intermediates
    metric_thr_var = pd.DataFrame(columns=['cmdb_id','kpi_name','global_p95','global_p5'])
    trace_thr_var  = pd.DataFrame(columns=['cmdb_id','trace_name','global_p95','global_p5'])
    df_metric_anom = pd.DataFrame(columns=df_metric.columns.tolist()+['global_p95','global_p5','kpi_name'])
    df_trace_anom  = pd.DataFrame(columns=df_trace.columns.tolist()+['global_p95','global_p5','trace_name'])

# Helper to get earliest anomaly timestamp or None
def earliest_ts_or_null(df, group_col):
    if df.empty:
        return {}
    res = df.groupby(group_col)['timestamp'].min().to_dict()
    return res

# 3) For each paymentservice cmdb_id compute required aggregates
rows = []
for cm in payment_cmdbs:
    # earliest metric anomaly ts
    metric_earliest = None
    trace_earliest = None
    metric_count_2080 = 0
    metric_count_2140 = 0
    trace_count_2080 = 0
    trace_count_2140 = 0
    total_metric_in_window = 0
    total_trace_in_window = 0

    if not df_metric_anom.empty:
        subm = df_metric_anom[df_metric_anom['cmdb_id']==cm]
        if not subm.empty:
            metric_earliest = int(subm['timestamp'].min())
            total_metric_in_window = int(subm.shape[0])
            metric_count_2080 = int(subm[subm['timestamp']==1647772080].shape[0])
            metric_count_2140 = int(subm[subm['timestamp']==1647772140].shape[0])

    if not df_trace_anom.empty:
        subt = df_trace_anom[df_trace_anom['cmdb_id']==cm]
        if not subt.empty:
            trace_earliest = int(subt['timestamp'].min())
            total_trace_in_window = int(subt.shape[0])
            trace_count_2080 = int(subt[subt['timestamp']==1647772080].shape[0])
            trace_count_2140 = int(subt[subt['timestamp']==1647772140].shape[0])

    rows.append({
        'cmdb_id': cm,
        'earliest_metric_anomaly_ts': metric_earliest if metric_earliest is not None else pd.NA,
        'earliest_trace_anomaly_ts': trace_earliest if trace_earliest is not None else pd.NA,
        'metric_anomaly_count_at_1647772080': metric_count_2080,
        'metric_anomaly_count_at_1647772140': metric_count_2140,
        'trace_anomaly_count_at_1647772080': trace_count_2080,
        'trace_anomaly_count_at_1647772140': trace_count_2140,
        'total_metric_anomalies_in_window': total_metric_in_window,
        'total_trace_anomalies_in_window': total_trace_in_window
    })

payment_summary = pd.DataFrame(rows, columns=[
    'cmdb_id','earliest_metric_anomaly_ts','earliest_trace_anomaly_ts',
    'metric_anomaly_count_at_1647772080','metric_anomaly_count_at_1647772140',
    'trace_anomaly_count_at_1647772080','trace_anomaly_count_at_1647772140',
    'total_metric_anomalies_in_window','total_trace_anomalies_in_window'
])

# 4) Return any rows from error_logs.csv and log.csv for these paymentservice cmdb_ids within the incident window
error_rows_payment = df_err[(df_err['cmdb_id'].isin(payment_cmdbs)) & (df_err['timestamp'] >= start_ts) & (df_err['timestamp'] <= end_ts)][['timestamp','cmdb_id','message']].copy()
error_rows_payment['timestamp'] = error_rows_payment['timestamp'].astype(int)

log_rows_payment = df_log[(df_log['cmdb_id'].isin(payment_cmdbs)) & (df_log['timestamp'] >= start_ts) & (df_log['timestamp'] <= end_ts)][['timestamp','cmdb_id','log_name','value']].copy()
log_rows_payment_sorted = log_rows_payment.sort_values('timestamp', ascending=True).head(50)
log_rows_payment_sorted['timestamp'] = log_rows_payment_sorted['timestamp'].astype(int)

# Display compact outputs: payment_summary (one row per payment cmdb), error_rows_payment, log_rows_payment_sorted (<=50)
payment_summary, error_rows_payment, log_rows_payment_sorted

```
Out[10]:
```
Summary of paymentservice anomalies around the peak minutes (March 20, 2022 18:00–18:30 UTC+8)

What was computed
- Global P95 and P5 were computed per metric KPI and per trace_name using the full series; constant series (global_p95 == global_p5) were excluded.
- Metric and trace points inside the 18:00–18:30 window were flagged when value >= global_p95 OR value <= global_p5.
- Results aggregated per paymentservice cmdb_id (paymentservice, paymentservice-0, paymentservice-1, paymentservice-2, paymentservice2-0).

Key compact results (one row per payment cmdb_id)
- paymentservice (service-level): earliest metric anomaly ts = 1647770400; no trace anomalies; small metric counts at the two peak minutes (metric anomalies at 1647772140 = 1); total_metric_anomalies_in_window = 13.
- paymentservice-0:
  - earliest_metric_anomaly_ts = 1647770400; earliest_trace_anomaly_ts = 1647770400
  - metric anomalies at 1647772080 = 17, at 1647772140 = 18
  - trace anomalies at 1647772080 = 0, at 1647772140 = 2
  - total_metric_anomalies_in_window = 388; total_trace_anomalies_in_window = 64
- paymentservice-1:
  - earliest_metric_anomaly_ts = 1647770400; earliest_trace_anomaly_ts = 1647770400
  - metric anomalies at 1647772080 = 16, at 1647772140 = 16
  - trace anomalies at 1647772080 = 2, at 1647772140 = 1
  - totals: metrics = 376, traces = 68
- paymentservice-2:
  - earliest_metric_anomaly_ts = 1647770400; earliest_trace_anomaly_ts = 1647770400
  - metric anomalies at 1647772080 = 18, at 1647772140 = 13
  - trace anomalies at 1647772080 = 4, at 1647772140 = 1
  - totals: metrics = 423, traces = 69
- paymentservice2-0:
  - earliest_metric_anomaly_ts = 1647770400; earliest_trace_anomaly_ts = 1647770580
  - metric anomalies at 1647772080 = 13, at 1647772140 = 18
  - trace anomalies at 1647772080 = 0, at 1647772140 = 0
  - totals: metrics = 426, traces = 15

Logs / error_logs within the window
- error_logs: no error_log rows for any paymentservice cmdb_id inside the 18:00–18:30 window (empty result).
- log.csv: many per-minute log rows for the paymentservice instances within the window (returned up to 50 rows). Observed pattern in printed samples: log.error_count generally 0; log.row_count present with small integer counts (varies per minute).

Interpretation & next steps
- All paymentservice instances show metric anomalies starting at the beginning of the window (earliest anomaly = 1647770400) and substantial total metric anomaly counts across the window (hundreds per instance). Trace anomalies are present but much fewer than metric anomalies for most instances.
- At the two previously-identified peak minutes (1647772080 and 1647772140), each payment instance contributed on the order of ~13–18 metric-anomaly points and a smaller number of trace anomalies (0–4 typically).
- No paymentservice error_log entries were found in the incident window, and log.error_count samples shown are 0 — so metric/trace anomalies did not coincide with explicit error logs for paymentservice in the provided data.
- Recommended follow-up:
  - Inspect the specific metric KPIs and trace names driving the high metric anomaly counts (per-instance lists are stored in kernel variables).
  - Correlate paymentservice traces at the peak minutes with checkout/frontend traces and the error entries at 1647772098 to see if paymentservice anomalies are upstream or downstream symptoms.
  - Check downstream integrations (e.g., external payment gateways) and resource metrics for paymentservice instances if specific KPIs indicate saturation or latency.

The original code execution output of IPython Kernel is also provided below for reference:

(             cmdb_id  earliest_metric_anomaly_ts earliest_trace_anomaly_ts  metric_anomaly_count_at_1647772080  metric_anomaly_count_at_1647772140  trace_anomaly_count_at_1647772080  trace_anomaly_count_at_1647772140  total_metric_anomalies_in_window  total_trace_anomalies_in_window
0     paymentservice                  1647770400                      <NA>                                   0                                   1                                  0                                  0                                13                                0
1   paymentservice-0                  1647770400                1647770400                                  17                                  18                                  0                                  2                               388                               64
2   paymentservice-1                  1647770400                1647770400                                  16                                  16                                  2                                  1                               376                               68
3   paymentservice-2                  1647770400                1647770400                                  18                                  13                                  4                                  1                               423                               69
4  paymentservice2-0                  1647770400                1647770580                                  13                                  18                                  0                                  0                               426                               15, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [],       timestamp           cmdb_id         log_name  value
36   1647770400  paymentservice-0  log.error_count    0.0
37   1647770400  paymentservice-0    log.row_count    6.0
38   1647770400  paymentservice-1  log.error_count    0.0
39   1647770400  paymentservice-1    log.row_count   12.0
40   1647770400  paymentservice-2  log.error_count    0.0
..          ...               ...              ...    ...
463  1647770820  paymentservice-1    log.row_count    9.0
464  1647770820  paymentservice-2  log.error_count    0.0
465  1647770820  paymentservice-2    log.row_count    7.0
522  1647770880  paymentservice-0  log.error_count    0.0
523  1647770880  paymentservice-0    log.row_count    5.0

[50 rows x 4 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[11]:

import pandas as pd
import numpy as np
import pytz
from datetime import datetime

# Ensure pandas will not truncate message column when displaying
pd.set_option('display.max_colwidth', None)

# Reuse existing dataframes if present
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv', dtype={'timestamp': int})
try:
    df_err
except NameError:
    df_err = pd.read_csv('error_logs.csv', dtype={'timestamp': int})

# Ensure integer timestamps
df_trace['timestamp'] = df_trace['timestamp'].astype(int)
df_err['timestamp'] = df_err['timestamp'].astype(int)

# 1) Error logs: rows at timestamp 1647772098 for frontend-2 and checkoutservice-1
err_rows = df_err[
    (df_err['timestamp'] == 1647772098) &
    (df_err['cmdb_id'].isin(['frontend-2', 'checkoutservice-1']))
][['timestamp','cmdb_id','message']].copy()
err_rows['timestamp'] = err_rows['timestamp'].astype(int)

# 2) Focused trace anomalies
# Targets and name filters
targets = ['frontend-2','checkoutservice-1','paymentservice-0','paymentservice-1','paymentservice-2','paymentservice2-0']
pattern = r'to_checkoutservice|to_paymentservice|from_frontend|to_payment'

# a) Identify matching trace series (full series) and compute global thresholds (P95,P5)
trace_candidates = df_trace[
    df_trace['trace_name'].str.contains(pattern, regex=True) &
    df_trace['cmdb_id'].isin(targets)
].copy()

# Compute global thresholds per (cmdb_id, trace_name) using full available series (rule 9)
trace_thresholds_focused = trace_candidates.groupby(['cmdb_id','trace_name'])['value'].agg(
    global_p95 = lambda x: float(np.percentile(x.values, 95)),
    global_p5  = lambda x: float(np.percentile(x.values, 5))
).reset_index()

# Exclude constant series (global_p95 == global_p5)
trace_thresholds_var = trace_thresholds_focused[trace_thresholds_focused['global_p95'] != trace_thresholds_focused['global_p5']].copy()

# b) Filter trace points to the 1647772040..1647772160 time range (±2 min around 1647772098)
start_ts = 1647772040
end_ts   = 1647772160

df_trace_window = df_trace[
    (df_trace['timestamp'] >= start_ts) &
    (df_trace['timestamp'] <= end_ts) &
    (df_trace['cmdb_id'].isin(targets)) &
    (df_trace['trace_name'].str.contains(pattern, regex=True))
].copy()

# Merge thresholds to keep only non-constant focused series
df_trace_win = df_trace_window.merge(trace_thresholds_var, on=['cmdb_id','trace_name'], how='inner')

# Mark anomaly points: value >= global_p95 OR value <= global_p5
df_trace_win['is_anom'] = (df_trace_win['value'] >= df_trace_win['global_p95']) | (df_trace_win['value'] <= df_trace_win['global_p5'])

# Keep anomaly points
df_trace_anom = df_trace_win[df_trace_win['is_anom']].copy()

# c) Group consecutive anomaly points into runs per trace series and produce compact table
if df_trace_anom.empty:
    trace_anom_head = pd.DataFrame(columns=[
        'cmdb_id','trace_name','anomaly_count','number_of_fault_runs',
        'earliest_fault_ts','latest_fault_ts','window_min_value','window_max_value','global_p95','global_p5'
    ])
else:
    # window-level min/max among all points in the selected time range (per series)
    window_stats = df_trace_window.merge(trace_thresholds_var, on=['cmdb_id','trace_name'], how='inner') \
                                  .groupby(['cmdb_id','trace_name'])['value'].agg(window_min_value='min', window_max_value='max') \
                                  .reset_index()

    # Function to summarize runs for anomaly points of a series
    def summarize_runs(g):
        g = g.sort_values('timestamp').reset_index(drop=True)
        # consecutive if diff == 60 seconds (per-minute resolution); treat others as breaks
        diffs = g['timestamp'].diff().fillna(999999)
        run_id = (diffs != 60).cumsum()
        runs = g.groupby(run_id)['timestamp'].agg(start_ts='min', end_ts='max').reset_index(drop=True)
        anomaly_count = int(g.shape[0])
        number_of_fault_runs = int(runs.shape[0])
        earliest_fault_ts = int(runs['start_ts'].min())
        latest_fault_ts = int(runs['end_ts'].max())
        return pd.Series({
            'anomaly_count': anomaly_count,
            'number_of_fault_runs': number_of_fault_runs,
            'earliest_fault_ts': earliest_fault_ts,
            'latest_fault_ts': latest_fault_ts
        })

    anom_summary = df_trace_anom.groupby(['cmdb_id','trace_name']).apply(summarize_runs).reset_index()

    # Merge with thresholds and window stats
    trace_anom_agg = anom_summary.merge(trace_thresholds_var, on=['cmdb_id','trace_name'], how='left') \
                                 .merge(window_stats, on=['cmdb_id','trace_name'], how='left')

    # Ensure integer timestamps
    trace_anom_agg['earliest_fault_ts'] = trace_anom_agg['earliest_fault_ts'].astype(int)
    trace_anom_agg['latest_fault_ts']   = trace_anom_agg['latest_fault_ts'].astype(int)

    # Final compact table ordered by anomaly_count desc, limit 50 rows
    trace_anom_head = trace_anom_agg[['cmdb_id','trace_name','anomaly_count','number_of_fault_runs',
                                      'earliest_fault_ts','latest_fault_ts','window_min_value','window_max_value',
                                      'global_p95','global_p5']].sort_values('anomaly_count', ascending=False).head(50)

# Display compact outputs: err_rows and trace_anom_head
err_rows, trace_anom_head

```
Out[11]:
```
Findings (focused on timestamp 1647772098 ± 2 minutes)

1) Error log rows (exact messages)
- 1647772098, frontend-2: "severity: error, message: request error"
- 1647772098, checkoutservice-1: "severity: warning, message: failed to send order confirmation to "someone@example.com": rpc error: code = Canceled desc = context canceled"

2) Focused trace anomalies (filtered trace_names containing to_checkoutservice / to_paymentservice / from_frontend / to_payment; window 1647772040–1647772160)

Top compact results (per trace series — shown only key columns)
- checkoutservice-1.trace.to_paymentservice-1.row_count
  - anomaly_count=2, runs=1, earliest=1647772080, latest=1647772140
  - window values: min=1, max=1; global_p95=2.0, global_p5=1.0
- checkoutservice-1.trace.from_frontend-1.row_count
  - anomaly_count=2, runs=1, earliest=1647772080, latest=1647772140
  - window min=1, max=2; global_p95=2.0, global_p5=1.0
- checkoutservice-1.trace.to_paymentservice-2.row_count
  - anomaly_count=2, runs=1, earliest=1647772080, latest=1647772140
  - window min=1, max=1; global_p95=2.0, global_p5=1.0
- checkoutservice-1.trace.from_frontend-1.duration_mean
  - anomaly_count=1, runs=1, earliest=latest=1647772080
  - window_min=0.042661, window_max=43.315353, global_p95=0.861655, global_p5=0.042110
  - Notes: a very large single duration sample (~43.3s) well above global p95 → clear latency spike at 1647772080
- checkoutservice-1.trace.from_frontend-1.duration_p95
  - anomaly_count=1, runs=1, earliest=latest=1647772080
  - window_max=43.315353, global_p95=0.861655
- checkoutservice-1.trace.from_frontend-2.row_count
  - anomaly_count=1 at 1647772080 (window 1–1; global_p95=1.1, p5=1.0)
- checkoutservice-1.trace.to_paymentservice-0.row_count
  - anomaly_count=1 at 1647772140 (window 1–1; global_p95=2.0, p5=1.0)
- checkoutservice-1.trace.to_paymentservice-2.duration_mean and duration_p95
  - each anomaly_count=1 at 1647772080; small duration values just at/above thresholds
- frontend-2.trace.to_checkoutservice-1.row_count
  - anomaly_count=1 at 1647772080 (window 1–1; global_p95=1.1, p5=1.0)

Interpretation and actionable notes
- The error log entries at 1647772098 (frontend-2 error; checkoutservice warning about failed order confirmation with rpc Canceled) coincide temporally with the trace anomalies cluster around 1647772080–1647772140.
- The most significant signal is a large latency outlier on checkoutservice-1.trace.from_frontend-1.duration_mean/p95 (single sample ~43.3s at 1647772080, far above historical p95 ≈ 0.86s). This strongly suggests a request latency spike at that time impacting checkoutservice.
- Multiple row_count anomalies and small-duration anomalies on checkoutservice→paymentservice traces indicate abnormal request behavior between checkout and payment services in the same window.
- frontend-2 shows only a small row_count anomaly (1 point) toward checkoutservice at 1647772080 but the error at 1647772098 on frontend-2 aligns with the checkoutservice issues.

Recommended next steps
- Inspect full trace spans (full trace tree) around timestamp 1647772080–1647772140 for checkoutservice-1 to identify root span and error propagation.
- Correlate with paymentservice traces (same window) to determine whether payment calls timed out or caused cancellations.
- Review network/mesh (Istio) and backend/payment gateway logs for timeouts or cancellations matching the RPC Canceled message.
- If desired, I can extract the raw trace records for any of the highlighted trace_names around 1647772080 for deeper inspection.

The original code execution output of IPython Kernel is also provided below for reference:

(    timestamp            cmdb_id                                                                                                                                     message
0  1647772098         frontend-2                                                                                                     severity: error, message: request error
1  1647772098  checkoutservice-1  severity: warning, message: failed to send order confirmation to "someone@example.com": rpc error: code = Canceled desc = context canceled,              cmdb_id                               trace_name  anomaly_count  number_of_fault_runs  earliest_fault_ts  latest_fault_ts  window_min_value  window_max_value  global_p95  global_p5
5  checkoutservice-1      trace.to_paymentservice-1.row_count              2                     1         1647772080       1647772140          1.000000          1.000000    2.000000   1.000000
2  checkoutservice-1          trace.from_frontend-1.row_count              2                     1         1647772080       1647772140          1.000000          2.000000    2.000000   1.000000
8  checkoutservice-1      trace.to_paymentservice-2.row_count              2                     1         1647772080       1647772140          1.000000          1.000000    2.000000   1.000000
0  checkoutservice-1      trace.from_frontend-1.duration_mean              1                     1         1647772080       1647772080          0.042661         43.315353    0.861655   0.042110
1  checkoutservice-1       trace.from_frontend-1.duration_p95              1                     1         1647772080       1647772080          0.043676         43.315353    0.861655   0.042110
3  checkoutservice-1          trace.from_frontend-2.row_count              1                     1         1647772080       1647772080          1.000000          1.000000    1.100000   1.000000
4  checkoutservice-1      trace.to_paymentservice-0.row_count              1                     1         1647772140       1647772140          1.000000          1.000000    2.000000   1.000000
6  checkoutservice-1  trace.to_paymentservice-2.duration_mean              1                     1         1647772080       1647772080          0.000151          0.000176    0.000176   0.000147
7  checkoutservice-1   trace.to_paymentservice-2.duration_p95              1                     1         1647772080       1647772080          0.000151          0.000176    0.000176   0.000147
9         frontend-2     trace.to_checkoutservice-1.row_count              1                     1         1647772080       1647772080          1.000000          1.000000    1.100000   1.000000)```
```

In [ ]:
# In[12]:

import pandas as pd
import pytz

# reuse df_trace if present, otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv', dtype={'timestamp': int})

# Ensure timestamp is integer
df_trace['timestamp'] = df_trace['timestamp'].astype(int)

# timezone variable for UTC+8 (per rule 10)
tz = pytz.timezone('Asia/Shanghai')

# Filter for checkoutservice-1 within ±20s of 1647772080 (i.e., 1647772060..1647772100)
start_ts = 1647772060
end_ts = 1647772100

df_checkout_window = df_trace[
    (df_trace['cmdb_id'] == 'checkoutservice-1') &
    (df_trace['timestamp'] >= start_ts) &
    (df_trace['timestamp'] <= end_ts)
][['timestamp','cmdb_id','trace_name','value']].sort_values('timestamp').copy()

# Ensure integer timestamps remain
df_checkout_window['timestamp'] = df_checkout_window['timestamp'].astype(int)

# Compact output (limit to 50 rows)
df_checkout_window.head(50)

```
Out[12]:
```
Summary of checkoutservice-1 trace rows in 1647772060–1647772100 (±20s around 1647772080)

- Returned rows (limited to 50) are all at timestamp 1647772080 (integer seconds).
- Key finding: trace.from_frontend-1.duration_mean and trace.from_frontend-1.duration_p95 are both 43.315353 seconds at 1647772080 — a very large latency spike compared to other duration values in this window.
- Other observations at 1647772080:
  - trace.from_frontend-1.row_count = 1 and trace.from_frontend-1.error_rate = 0.0.
  - Several from_frontend and to_paymentservice trace entries are present; most to_paymentservice duration values are small (fractions of a second) and row_count values are typically 1.
  - Multiple trace.series for paymentservice calls (to_paymentservice-0/1/2) appear at the same timestamp, some with small duration_p95/mean values.
- Interpretation: A clear, single large request latency occurred at checkoutservice-1 at 1647772080 (≈43.3s) for requests coming from frontend-1. Other traces at the same second show normal/small durations and minimal row counts.
- Suggested next steps: inspect the full trace spans for trace.from_frontend-1 at 1647772080, correlate with logs and paymentservice spans around that second, and review error logs around 1647772098 for related failures.

The original code execution output of IPython Kernel is also provided below for reference:

timestamp            cmdb_id                               trace_name      value
32480  1647772080  checkoutservice-1      trace.from_frontend-1.duration_mean  43.315353
32481  1647772080  checkoutservice-1       trace.from_frontend-1.duration_p95  43.315353
32482  1647772080  checkoutservice-1         trace.from_frontend-1.error_rate   0.000000
32483  1647772080  checkoutservice-1          trace.from_frontend-1.row_count   1.000000
32484  1647772080  checkoutservice-1      trace.from_frontend-2.duration_mean   0.268460
...           ...                ...                                      ...        ...
32525  1647772080  checkoutservice-1   trace.to_paymentservice-1.duration_p95   0.000147
32526  1647772080  checkoutservice-1     trace.to_paymentservice-1.error_rate   0.000000
32527  1647772080  checkoutservice-1      trace.to_paymentservice-1.row_count   1.000000
32528  1647772080  checkoutservice-1  trace.to_paymentservice-2.duration_mean   0.000176
32529  1647772080  checkoutservice-1   trace.to_paymentservice-2.duration_p95   0.000176

[50 rows x 4 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```